# RAG — Complete Class Notes (Groq Version)
### Ingestion → Retrievers → Query Transformation → Post-Retrieval → Generation → Agentic RAG

**Paper used for practice:** *Attention Is All You Need* (Vaswani et al., 2017) — the Transformer paper (15 pages)

**What changed from the class notebooks**
- **LLM:** OpenAI `ChatOpenAI` → **Groq `ChatGroq`** (model: `openai/gpt-oss-120b`)
- **Embeddings:** OpenAI embeddings → **free local HuggingFace embeddings**, because Groq does **not** provide an embedding model.
- **One vector store for everything:** the PDF is embedded **once** (Part B). Every retriever, query technique, reranker and agent below reuses the same `vector_store`, `chunks` and `bm25_retriever`. No dummy documents, no new Chroma collection per topic.
- **Cohere reranker removed** (needs a paid/extra key). We use the free local **cross-encoder** and **FlashRank** rerankers instead.
- **Tavily web search is optional.** If `TAVILY_API_KEY` is not set, the agents simply work with the paper only.
- Where the two class notebooks taught the same idea (MultiQuery, HyDE, Ensemble, reranking), only the **better version** is kept.

---

## Contents (the order follows the RAG pipeline)

| Part | Stage | Topics |
|---|---|---|
| **A** | Setup | 1. Install · 2. Imports · 3. API keys · 4. Groq LLM · 5. Helper functions |
| **B** | Ingestion (PDF → vector store) | 6. Load PDF · 7. Metadata · 8. Chunking · 9. Embeddings · 10. Vector store |
| **C** | Retrievers | 11. What is a retriever · 12. Similarity · 13. MMR · 14. Score threshold · 15. Distance metrics · 16. Metadata filtering · 17. Pre vs post filtering · 18. Sparse (BM25) · 19. Dense · 20. Hybrid / Ensemble (RRF) · 21. Parent Document · 22. Custom Parent Document · 23. Self-Query · 24. Custom Self-Query |
| **D** | Query transformation (before retrieval) | 25. Query rewriting · 26. Query expansion (custom MultiQuery) · 27. MultiQueryRetriever (built-in) · 28. RAG Fusion · 29. Query decomposition · 30. HyDE (custom) · 31. HyDE (built-in) |
| **E** | Post-retrieval (after retrieval) | 32. Contextual compression (LLM extractor) · 33. Embeddings filter & pipeline · 34. Custom contextual compression · 35. Cross-encoder reranking · 36. FlashRank reranking · 37. Hybrid + reranking |
| **F** | Generation | 38. Augmentation + generation · 39. Comparing retrievers end-to-end |
| **G** | Agentic RAG (LangGraph) | 40. LangGraph basics & shared pieces · 41. Base RAG graph · 42. Conditional retrieval · 43. Tool-use retrieval · 44. Multi-round retrieval limit · 45. Evaluation & query rewriting · 46. Query decomposition · 47. Full agentic RAG |
| **H** | Wrap-up | 48. Summary · 49. Homework |

```
PDF ──► Load ──► Metadata ──► Chunks ──► Embeddings ──► Vector store        (Part B, done once)
                                                            │
User query ──► [Query transformation] ──► Retriever ──► [Compression / Rerank] ──► LLM ──► Answer
                    (Part D)                (Part C)            (Part E)          (Part F)
                                                            │
                                        Agent decides what to do in a loop          (Part G)
```

---

# RAG Techniques — Roman Urdu Summary(concepts)

*Paper: Attention Is All You Need · LLM: Groq (`openai/gpt-oss-120b`) · Embeddings: HuggingFace MiniLM*

## Part A — Setup

| Concept | Explanation |
|---|---|
| **Groq LLM (`ChatGroq`)** | Groq ka chat model jo bohat fast jawab deta hai. Poori notebook mein ek hi `llm` banaya aur har jagah wahi use kiya. `temperature=0` ka matlab hai ke same sawal par (taqreeban) same jawab aaye. |
| **Embedding model kyun HuggingFace** | Groq sirf chat models deta hai, embedding model nahi. Is liye free local model `all-MiniLM-L6-v2` use kiya jo apne computer par chalta hai. |
| **Structured output** | `llm.with_structured_output(PydanticClass)` LLM ko majboor karta hai ke free text ki jagah ek saaf Python object (list, bool waghera) wapas de. Is tarah output ko khud parse nahi karna parta. |
| **Helper functions** | `display_documents`, `show_ids`, `deduplicate_documents`, `format_docs` chhote functions hain. Yeh results ko saaf print karte hain, duplicate chunks hataate hain, aur chunks ko page label ke saath ek context string mein jor dete hain. |

## Part B — Ingestion (PDF → Vector Store)

| Concept | Explanation |
|---|---|
| **Document Loader (`PyPDFLoader`)** | PDF ko parh kar har page ka ek `Document` banata hai. Har Document mein `page_content` (text) aur `metadata` (source, page number) hota hai. |
| **Custom Metadata** | Har page par apne labels lagana, jaise section, paper_page, year, organization. Inhi labels se baad mein filtering, self-query aur citations mumkin hote hain. |
| **Chunking (`RecursiveCharacterTextSplitter`)** | Bare pages ko chhote tukron (chunks) mein kaatna. `chunk_size=1000` max size hai aur `chunk_overlap=200` ka matlab hai ke saath wale chunks 200 characters share karte hain, taake kinare par kata hua jumla zaya na ho. Pehle paragraph, phir line, phir space par todta hai. |
| **Embeddings** | Text ko numbers ki list (vector) mein badalna. Jin texts ka matlab milta julta ho, unke vectors qareeb hote hain. MiniLM har text ke 384 numbers banata hai. |
| **Normalization** | `normalize_embeddings=True` har vector ki length 1 kar deta hai. Is se cosine similarity aur dot product ka result same aata hai. |
| **Vector Store (Chroma)** | Database jo har chunk ko uske vector ke saath save karta hai aur query ke qareeb tareen chunks jaldi dhoondta hai. `persist_directory` se disk par save hota hai, taake baar baar embed na karna pare. |
| **HNSW + cosine space** | HNSW Chroma ka fast search index hai. `space: cosine` ka matlab hai ke doori cosine se naapi jayegi. |
| **REBUILD_INDEX flag** | `True` ho to purana database delete karke naya banta hai. `False` ho to saved database load hota hai, jo zyada fast hai. |

## Part C — Retrievers

| Concept | Explanation |
|---|---|
| **Retriever** | Ek simple interface: sawal do, relevant Documents ki list wapas lo (`.invoke(query)`). Har retriever ka interface same hai, is liye search ka tareeqa badalne par baqi code nahi badalta. |
| **Custom Retriever (`BaseRetriever`)** | Apna retriever banane ke liye `BaseRetriever` se inherit karo aur logic `_get_relevant_documents()` mein likho. Phir yeh chains aur LangGraph mein baaqi retrievers ki tarah chalta hai. |
| **Similarity Search** | Query ka vector bana kar sab se qareeb `k` chunks wapas karta hai. Kamzori: top results aksar ek hi baat dohraate hain. |
| **MMR (Maximal Marginal Relevance)** | Pehle zyada candidates (`fetch_k`) laata hai, phir aise chunks chunta hai jo relevant bhi hon aur ek doosre se mukhtalif bhi. `lambda_mult` 1 ke qareeb ho to relevance zyada, 0 ke qareeb ho to diversity zyada. |
| **Relevance Score** | 0 se 1 tak number jo batata hai chunk kitna relevant hai (zyada behtar). Cosine mein `relevance = 1 − distance`. |
| **Score-Threshold Retriever** | Sirf woh chunks deta hai jinka score threshold se upar ho. Koi achha chunk na mile to kuch nahi deta, taake LLM ko kachra na jaye. Threshold har embedding model ke liye alag set karna parta hai. |
| **Cosine Similarity** | Do vectors ke darmiyan angle naapta hai, sirf direction (matlab) dekhta hai. Zyada value behtar. |
| **Euclidean Distance** | Do vectors ke darmiyan seedhi doori. Kam value behtar. |
| **Dot Product** | Vectors ko multiply karke jorna. Normalized vectors par yeh cosine ke barabar hota hai. Teeno metrics ranking same deti hain jab vectors normalized hon. |
| **Metadata Filtering** | Sirf un chunks mein search karna jinki metadata condition se milti ho, jaise `section = training`. Chroma operators: `$eq, $ne, $gt, $gte, $lt, $lte, $in, $nin, $and, $or`. |
| **Pre-filtering** | Filter search se pehle lagta hai: database sirf matching chunks mein dhoondta hai. Hamesha `k` results milte hain (agar mojood hon) aur yeh fast hai. |
| **Post-filtering** | Pehle sab mein search hoti hai, phir Python mein filter lagta hai. Results `k` se kam reh sakte hain. Complex conditions ke liye kaam aata hai. |
| **Sparse Retrieval (BM25)** | Keyword search: matlab nahi samajhta, asal alfaaz match karta hai. Kam milne wale alfaaz (jaise "BLEU", "WMT") ko zyada wazan deta hai. Exact terms, codes aur numbers mein strong, synonyms mein kamzor. Embeddings ki zaroorat nahi. |
| **Dense Retrieval** | Embeddings se matlab ki bunyaad par search. Synonyms aur paraphrase samajhta hai, lekin exact rare keywords ya IDs miss kar sakta hai. |
| **Hybrid Retrieval / Ensemble Retriever** | BM25 aur dense dono chala kar unke results ko ek ranking mein milana. Keyword ki taaqat bhi milti hai aur matlab ki bhi. Similarity + MMR ko bhi isi tarah mila sakte hain. |
| **RRF (Reciprocal Rank Fusion)** | Results milane ka formula jo sirf rank position dekhta hai, raw scores nahi: `weight / (60 + rank)`. Jo chunk dono lists mein upar ho, woh jeet jata hai. |
| **Parent Document Retriever** | Search chhote child chunks (400 chars) par hoti hai, jo precise hai, lekin wapas bare parent chunks (1500 chars) milte hain, jin mein poora context hota hai. Children vector store mein aur parents docstore mein rehte hain. |
| **InMemoryStore vs LocalFileStore** | Parents rakhne ki jagah. InMemoryStore RAM mein hai aur restart par khatam ho jata hai. LocalFileStore + `create_kv_docstore` disk par JSON files mein save karta hai. |
| **Custom Parent Document Retriever** | Wahi parent-child idea khud likha: parent ko `uuid` do, child par parent id tag karo, `mset` se save karo, search ke baad `mget` se parents nikaalo. |
| **Self-Query Retriever** | LLM sawal parh kar usay do hisson mein todta hai: semantic query (search ke liye) aur metadata filter. Jaise "training section mein regularization" → query "regularization" + filter `section = training`. `ChromaTranslator` filter ko Chroma ki syntax mein badalta hai. |
| **AttributeInfo** | LLM ko batata hai ke kaun kaun se metadata fields hain, unki type kya hai aur allowed values kya hain, taake woh sahi filter bana sake. |
| **Custom Self-Query Retriever** | Pydantic schema (`SelfQuerySchema`, `MetadataFilter`) aur structured output se khud bana hua self-query. Isme number fields ko text se int mein badalna ("8" → 8) aur na-mojood fields ignore karna bhi shamil hai. |

## Part D — Query Transformation (Search se pehle)

| Concept | Explanation |
|---|---|
| **Query Rewriting** | Chat mein follow-up sawal ("aur decoder us se kaise alag hai?") mein "us" ka matlab retriever nahi samajhta. LLM chat history dekh kar isay mukammal, akela (standalone) search query bana deta hai. |
| **Query Expansion** | User ke alfaaz paper ke alfaaz se mukhtalif ho sakte hain ("words ka order" vs "positional encoding"). LLM ek sawal ke kai versions banata hai (synonyms, technical terms), har ek se search hoti hai, phir results mila kar duplicates hataaye jaate hain. Is se recall barhta hai. |
| **Custom MultiQuery Retriever** | Query expansion ko ek custom retriever class mein band kiya: expand → har query se retrieve → unique union. Generated queries `last_queries` mein dekh sakte hain. |
| **MultiQueryRetriever (built-in)** | LangChain ka ready-made query expansion. `include_original=True` asal sawal se bhi search karta hai, taake asal alfaaz se match hone wale docs miss na hon. |
| **RAG Fusion** | MultiQuery jaisa hai (kai queries, har ek se search), lekin results ko sirf jorta nahi, balke RRF se rank karta hai. Jo chunk kai queries mein upar aaye, woh top par jata hai. Is liye output ek ranked top-n list hoti hai. |
| **Query Decomposition** | Ek complex sawal jo kai cheezein ek saath poochta ho, usay LLM chhote, alag sub-questions mein todta hai. Har sub-question se alag search hoti hai, phir saara evidence mila diya jata hai. Expansion mein same sawal naye alfaaz mein hota hai, decomposition mein alag alag chhote sawal. |
| **HyDE (Hypothetical Document Embeddings)** | Chhota sawal aur lamba technical paragraph alag dikhte hain. LLM pehle ek fake jawab paragraph likhta hai, aur usi se search hoti hai, kyunke fake jawab asal paper ke text jaisa dikhta hai. Fake jawab sirf search ke liye hai, final jawab hamesha asal documents se banta hai. |
| **Custom HyDE Retriever** | HyDE khud likha `BaseRetriever` se. Fake passage `last_hypothetical_document` mein save hota hai taake check kar sakein. |
| **HypotheticalDocumentEmbedder (built-in HyDE)** | Yeh retriever nahi, embeddings wrapper hai: query embed karne se pehle LLM se fake document likhwata hai. Same Chroma collection ke saath sirf query side par use hota hai. Index banate waqt kabhi use nahi karna. |

## Part E — Post-Retrieval (Search ke baad, LLM se pehle)

| Concept | Explanation |
|---|---|
| **ContextualCompressionRetriever** | Base retriever (candidates dhoondta hai) + compressor (unhein saaf, filter ya reorder karta hai). Compression aur reranking dono isi se hote hain. |
| **LLMChainExtractor** | Har retrieved chunk se LLM sirf relevant hissa jaisa ka taisa nikaalta hai. Kuch relevant na ho to `NO_OUTPUT` aata hai aur chunk drop ho jata hai. Context saaf ho jata hai, lekin har chunk par ek LLM call lagti hai, jo slow aur mehenga hai. |
| **EmbeddingsFilter** | Sasta filter, LLM nahi lagta. Query aur chunk ki similarity threshold se kam ho to poora chunk hata deta hai. Score `document.state["query_similarity_score"]` mein milta hai. MiniLM ke liye threshold ~0.45 rakha. |
| **DocumentCompressorPipeline** | Kai compressors ek line mein: pehle sasta (EmbeddingsFilter) kamzor chunks hataata hai, phir mehenga (LLM extractor) sirf bache hue chunks par chalta hai. Is se LLM calls kam hoti hain. |
| **Custom Contextual Compression** | LLM extractor aur compression retriever khud likhe: Step 1 retrieve, Step 2 compress. Original metadata compressed chunks par bhi rehta hai, taake citations chalti rahen. |
| **Reranking (Two-stage retrieval)** | Stage 1 mein fast search se zyada candidates (jaise 20) aate hain. Stage 2 mein ek zyada accurate model unhein dobara rank karta hai aur sirf best 5 rakhta hai. |
| **Bi-encoder vs Cross-encoder** | Bi-encoder (embeddings) query aur chunk ko alag alag embed karta hai: bohat fast, lakhon chunks ke liye. Cross-encoder query aur chunk ko ek saath parhta hai: slow lekin zyada accurate, chhoti list reorder karne ke liye. |
| **Cross-Encoder Reranker** | Free local model `ms-marco-MiniLM-L6-v2` jo har (query, chunk) pair ko score karke top_n rakhta hai. Cohere Rerank (paid API) ki jagah yeh use kiya. |
| **FlashRank Reranker** | Ek aur free local reranker. Chhote, optimized (ONNX) models ki wajah se CPU par bohat fast hai. Score `metadata["relevance_score"]` mein aata hai. |
| **Hybrid + Reranking (best pipeline)** | BM25 + Dense → RRF se candidates → cross-encoder se final top chunks. Production RAG mein yeh setup bohat common hai. |

## Part F — Generation

| Concept | Explanation |
|---|---|
| **Augmentation** | Retrieved chunks ko ek context string mein jor kar prompt mein daalna (`format_docs` page labels bhi lagata hai). |
| **Grounded Generation** | LLM ko kaha jata hai ke sirf context se jawab de, pages cite kare ([Page 5]), aur jawab na mile to "I don't know" kahe. Is se hallucination kam hoti hai. |
| **LCEL Chain** | `{"context": retriever \| format_docs, "question": RunnablePassthrough()} \| prompt \| llm \| StrOutputParser()` poori RAG pipeline ek line mein. `RunnablePassthrough` sawal ko waisa hi aage bhejta hai. |
| **Comparing Retrievers End-to-End** | Har retriever ka interface same hai, is liye kisi bhi retriever ko same RAG chain mein laga kar jawab compare kar sakte hain (`build_rag_chain`). |

## Part G — Agentic RAG (LangGraph)

| Concept | Explanation |
|---|---|
| **Agentic RAG** | Fixed pipeline (hamesha retrieve → generate) ki jagah LLM khud faisla karta hai: search karni hai ya nahi, kaun sa tool, dobara search karni hai ya nahi, sawal todna hai ya nahi. |
| **State** | Ek dict jo saare nodes share karte hain (query, context, generation waghera). States ek doosre se inherit karti hain, is liye har topic mein state barhti dikhti hai. |
| **Node** | Ek Python function jo state parhta hai aur updates ka dict wapas karta hai. |
| **Edge / Conditional Edge** | Edge fixed teer hai: A ke baad hamesha B. Conditional edge ek function hai jo state dekh kar agle node ka naam batata hai. |
| **MessagesState** | Ready-made state jisme `messages` list hoti hai. Naye messages replace nahi hote, add hote hain. Tool calling ke liye zaroori hai. |
| **Type hint wali baat** | Node ko sirf ussi state class ke fields milte hain jo uske type hint mein likhi ho. Isi liye har field jo node parhta hai, us class mein declare hona chahiye. |
| **Base RAG Graph** | Sab se simple graph: `START → retrieve → generate → END`. |
| **Conditional Retrieval** | Router node (`RouteDecision`) LLM se poochta hai ke sawal ko document chahiye ya general knowledge kaafi hai. "Pakistan ka capital?" par search skip ho jati hai. |
| **Tools (`@tool`)** | Functions jinhein LLM call kar sakta hai: `vector_store_search` (paper) aur `web_search` (Tavily, optional). `content_and_artifact` text LLM ko deta hai aur asal Documents artifact mein rakhta hai. |
| **bind_tools / ToolNode** | `llm.bind_tools(tools)` se LLM text ki jagah tool call wapas de sakta hai. `ToolNode` requested tools chala kar `ToolMessage` wapas karta hai. |
| **Agent Loop** | `agent → tools → agent → ...` jab tak LLM aur tool calls karta rahe. Phir `collect_tool_output` aakhri round ke documents jama karke context banata hai. |
| **Retrieval Limit** | Har tools round ke baad `retrieval_count` barhta hai. `max_retrieval_steps` par loop rok diya jata hai, taake API calls zaya na hon. |
| **Retrieval Evaluation (Corrective RAG)** | `evaluate_docs` mein LLM har document ko grade karta hai (relevant hai ya nahi). Sirf relevant documents context mein jaate hain. |
| **Query Rewriting (agent mein)** | Koi document relevant na nikle to LLM behtar query banata hai (zyada specific, alag alfaaz) aur agent dobara search karta hai. |
| **Fallback Answer** | `MAX_REWRITES` koshishon ke baad bhi kuch na mile to andaza lagane ki jagah seedha "relevant information nahi mili" ka jawab. |
| **Query Decomposition (agent mein)** | Agent khud check karta hai (`DecompositionCheck`) ke sawal kai hisson wala hai ya nahi. Agar hai to sub-queries bana kar `retrieval_task` mein "Step 1, Step 2" likhta hai. User ka asal `query` waisa hi rehta hai. |
| **Dynamic Prompt (`build_prompt`)** | Final system prompt run mein jo hua uske hisaab se banta hai: sawal toda gaya tha, query rewrite hui thi, context paper se aaya ya web se ya dono, web sources ki list, ya fallback. |
| **Full Agentic RAG** | Sab kuch ek graph mein: route → decomposition → agent ⇄ tools → limit → collect → evaluate → rewrite → build_prompt → generate. |

---
# Part A — Setup

## Topic 1: Install packages
Run this cell once. After it finishes, restart the kernel.

| Package | Why we need it |
|---|---|
| `langchain-groq` | Groq chat model (`ChatGroq`) |
| `langchain-huggingface` + `sentence-transformers` | Free local embeddings and the cross-encoder reranker |
| `langchain-chroma` | Chroma vector database |
| `langchain-community` | `PyPDFLoader`, `BM25Retriever`, `HuggingFaceCrossEncoder`, `FlashrankRerank`, `ChromaTranslator` |
| `langchain-classic` | `EnsembleRetriever`, `MultiQueryRetriever`, `ContextualCompressionRetriever`, `ParentDocumentRetriever`, `SelfQueryRetriever`, HyDE |
| `langgraph` | Graphs for agentic RAG (Part G) |
| `pypdf` | Reads PDF files |
| `rank_bm25` | The BM25 keyword-search algorithm |
| `flashrank` | Small, fast local reranker (Topic 36) |
| `lark` | Needed by the built-in `SelfQueryRetriever` to parse filters |
| `tavily-python` | Optional web-search tool for the agents |

In [ ]:
# !uv pip install -U langchain langchain-classic langchain-community langchain-groq langchain-huggingface langchain-chroma langgraph sentence-transformers pypdf rank_bm25 flashrank lark tavily-python pandas numpy

## Topic 2: Imports
All imports are kept in one place so the notebook runs from top to bottom without errors.

> Tip: `langchain-community` may show a *DeprecationWarning*. It is only a warning; the code still works.

In [ ]:
import getpass
import os
import shutil
import uuid
from pathlib import Path
from typing import Any, List, Literal, Optional

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field

# Loading and chunking
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings and vector store
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# LLM (Groq) and chains
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Retrievers
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import (
    EnsembleRetriever,
    MultiQueryRetriever,
    ParentDocumentRetriever,
    SelfQueryRetriever,
    ContextualCompressionRetriever,
)
from langchain_classic.storage import InMemoryStore, LocalFileStore, create_kv_docstore
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_community.query_constructors.chroma import ChromaTranslator
from langchain_classic.chains.hyde.base import HypotheticalDocumentEmbedder

# Post-retrieval: compression and reranking
from langchain_classic.retrievers.document_compressors import (
    LLMChainExtractor,
    EmbeddingsFilter,
    DocumentCompressorPipeline,
    CrossEncoderReranker,
)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.document_compressors import FlashrankRerank

# Agentic RAG (LangGraph)
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode

print("All imports loaded.")

## Topic 3: API keys
- **Groq (required):** free key from **console.groq.com → API Keys**.
- **Tavily (optional):** free key from **tavily.com**. It is only used as a *web search tool* in the agentic RAG part. Just press **Enter** to skip it — the agents will then search only the paper.

Each key is asked only if it is not already set in your environment.

In [ ]:
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

if not os.environ.get("TAVILY_API_KEY"):
    tavily_key = getpass.getpass("Enter your Tavily API key (optional, press Enter to skip): ")
    if tavily_key.strip():
        os.environ["TAVILY_API_KEY"] = tavily_key.strip()

TAVILY_AVAILABLE = bool(os.environ.get("TAVILY_API_KEY"))

print("Groq API key configured.")
print("Tavily web search:", "enabled" if TAVILY_AVAILABLE else "disabled (agents will use the paper only)")

## Topic 4: Groq LLM Setup
Many techniques in this notebook use an LLM: query rewriting, expansion, decomposition, HyDE, self-query, compression, generation and the agents. So we create the LLM once, here, and reuse the same `llm` everywhere.

- **Model:** `openai/gpt-oss-120b` (Groq's recommended production model; the older `llama-3.3-70b-versatile` was shut down in August 2026). It supports **structured output** and **tool calling**, which we need later.
- **`temperature=0`** → same input gives (almost) the same output. Good for search tasks.
- You can change the model with the `GROQ_CHAT_MODEL` environment variable. `openai/gpt-oss-20b` is a smaller, cheaper option.

> **Groq free tier:** it has a requests-per-minute limit. If you get a *rate limit* error, wait a few seconds and run the cell again.

In [ ]:
CHAT_MODEL = os.environ.get("GROQ_CHAT_MODEL", "openai/gpt-oss-120b")

llm = ChatGroq(
    model=CHAT_MODEL,
    temperature=0,
)

print("Chat model:", CHAT_MODEL)
print("Test reply:", llm.invoke("Reply with one word: ready").content)

## Topic 5: Helper functions
Small helpers used across the whole notebook:

| Helper | What it does |
|---|---|
| `display_documents()` | prints retrieved documents neatly (page, section, chunk id, text) |
| `show_ids()` | short view: only page, section and chunk id — good for comparing retrievers |
| `deduplicate_documents()` | removes repeated chunks (the same chunk is often found by several queries), keeps the order |
| `format_docs()` | joins documents into one context string with page labels, ready for the LLM |

In [ ]:
def display_documents(
    documents,
    title: str = "Retrieved Documents",
    max_documents: int = 10,
    max_characters: int = 500,
) -> None:
    # Print retrieved LangChain Documents in a readable way.
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)

    if not documents:
        print("No documents were returned.")
        return

    for rank, document in enumerate(documents[:max_documents], start=1):
        metadata = document.metadata
        extra = ""
        if "relevance_score" in metadata:
            extra = f" | Score: {float(metadata['relevance_score']):.4f}"
        if "rrf_score" in metadata:
            extra = f" | RRF score: {metadata['rrf_score']:.4f}"
        print(f"\nRANK: {rank}")
        print(f"Page: {metadata.get('paper_page')} | Section: {metadata.get('section')} | Chunk: {metadata.get('chunk_id')}{extra}")
        print("-" * 90)
        print(document.page_content[:max_characters])


def show_ids(documents, title: str) -> None:
    # Short view: only page, section and chunk id.
    print(f"\n{title}")
    for rank, document in enumerate(documents, start=1):
        print(rank, document.metadata.get("paper_page"), document.metadata.get("section"), document.metadata.get("chunk_id"))


def deduplicate_documents(documents):
    # Remove duplicate chunks while keeping their order.
    unique_documents = []
    seen_keys = set()

    for document in documents:
        key = document.metadata.get("chunk_id") or document.page_content
        if key not in seen_keys:
            seen_keys.add(key)
            unique_documents.append(document)

    return unique_documents


def format_docs(documents) -> str:
    # Join documents into one context string. Page labels let the LLM cite its sources.
    return "\n\n".join(
        f"[Page {d.metadata.get('paper_page', '?')} | {d.metadata.get('section', 'unknown')}]\n{d.page_content}"
        for d in documents
    )


print("Helper functions ready.")

---
# Part B — Ingestion (PDF → Vector Store)

This is the **indexing** half of RAG. It runs **once**; everything after Part B only *reads* from the vector store.

```
PDF file ──► Load (pages) ──► Add metadata ──► Split into chunks ──► Embed ──► Store in Chroma
```

## Topic 6: Loading the PDF

**What:** A *document loader* reads a file and turns it into LangChain `Document` objects.

**How `PyPDFLoader` works:** it creates **one `Document` per PDF page**. Each `Document` has:
- `page_content` → the text of that page
- `metadata` → extra info such as `source` (file path) and `page` (page index, starting from 0)

The cell below first looks for `attention_is_all_you_need.pdf` inside `DATA_DIR`. If the name is slightly different, it picks the only PDF in the folder automatically. **Change `DATA_DIR` to your own folder.**

In [ ]:
DATA_DIR = Path(
    r"C:\Users\abubakar\Desktop\agentic_notebook\00_data\parsing_data\data"
)

preferred_pdf = DATA_DIR / "attention_is_all_you_need.pdf"

if preferred_pdf.exists():
    PDF_PATH = preferred_pdf
else:
    available_pdfs = sorted(DATA_DIR.glob("*.pdf"))

    if len(available_pdfs) == 1:
        PDF_PATH = available_pdfs[0]
    elif len(available_pdfs) == 0:
        raise FileNotFoundError(f"No PDF file was found inside:\n{DATA_DIR}")
    else:
        raise RuntimeError(
            "Multiple PDF files were found. Please set PDF_PATH manually.\n"
            + "\n".join(str(path) for path in available_pdfs)
        )

print("PDF found:", PDF_PATH)

In [ ]:
loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()

print(f"Total PDF pages loaded: {len(pages)}")

print("\nFirst-page metadata:")
print(pages[0].metadata)

print("\nFirst 500 characters:")
print(pages[0].page_content[:500])

## Topic 7: Adding Custom Metadata

**What:** We attach our own labels (metadata) to every page, e.g. which **section** of the paper it belongs to.

**Why it matters:**
1. **Filtering** — later we can search *only* inside the `model_architecture` section (Topic 16).
2. **Self-query** — an LLM can turn *"what does the training section say…"* into a filter automatically (Topics 23–24).
3. **Citations** — we can show the user the page and section an answer came from (Part F).
4. **Access control** — a field like `access_level` can hide private documents.

**How:** We use the page number to decide the section. `PyPDFLoader` counts pages from 0, so we add 1 to get the real page number.

| Pages | Section in *Attention Is All You Need* |
|---|---|
| 1 | `front_matter` (title, authors, abstract) |
| 2 | `introduction` (1 Introduction, 2 Background) |
| 3–5 | `model_architecture` (3 Model Architecture: encoder/decoder, attention, multi-head attention) |
| 6 | `why_self_attention` (3.5 positional encoding, 4 Why Self-Attention) |
| 7 | `training` (5 Training: data, hardware, optimizer, regularization) |
| 8–9 | `results` (6 Results: translation, model variations, constituency parsing) |
| 10 | `conclusion` (7 Conclusion + start of references) |
| 11–12 | `references` |
| 13–15 | `attention_visualizations` (appendix figures) |

> Pages are not perfectly clean (page 6 holds the end of section 3 *and* section 4), but page-based labels are good enough for filtering and citations.

In [ ]:
def identify_section(paper_page: int) -> str:
    # Return the section name of the "Attention Is All You Need" paper for a given page number.
    if paper_page == 1:
        return "front_matter"
    if paper_page == 2:
        return "introduction"
    if 3 <= paper_page <= 5:
        return "model_architecture"
    if paper_page == 6:
        return "why_self_attention"
    if paper_page == 7:
        return "training"
    if 8 <= paper_page <= 9:
        return "results"
    if paper_page == 10:
        return "conclusion"
    if 11 <= paper_page <= 12:
        return "references"
    if 13 <= paper_page <= 15:
        return "attention_visualizations"
    return "unknown"


for page_document in pages:
    paper_page = int(page_document.metadata.get("page", 0)) + 1  # 0-based -> real page number

    page_document.metadata.update(
        {
            "paper": "Attention Is All You Need",
            "organization": "Google",
            "year": 2017,
            "document_type": "research_paper",
            "paper_page": paper_page,
            "section": identify_section(paper_page),
            "access_level": "public",
        }
    )

for page_document in pages:
    print(page_document.metadata["paper_page"], page_document.metadata["section"])

## Topic 8: Chunking (Splitting Text)

**What:** We cut each page into smaller pieces called **chunks**.

**Why:** One page is too big. A small chunk contains one focused idea, so search becomes more accurate, and we only send relevant text to the LLM.

**`RecursiveCharacterTextSplitter` settings:**
| Setting | Meaning |
|---|---|
| `chunk_size=1000` | Each chunk is at most ~1000 characters |
| `chunk_overlap=200` | Neighbouring chunks share 200 characters, so a sentence cut at the edge is not lost |
| `add_start_index=True` | Saves where the chunk starts inside the page (`start_index`) |

"Recursive" means it first tries to split on paragraphs (`\n\n`), then lines (`\n`), then spaces, so it keeps sentences together as much as possible.

Every chunk **keeps the metadata of its page**, and we also give each chunk a unique `chunk_id`.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = text_splitter.split_documents(pages)

for chunk_number, chunk in enumerate(chunks):
    paper_page = chunk.metadata.get("paper_page", "unknown")
    chunk.metadata["chunk_id"] = f"attention-page-{paper_page}-chunk-{chunk_number}"

print(f"Total pages:  {len(pages)}")
print(f"Total chunks: {len(chunks)}")
print("\nExample chunk metadata:")
print(chunks[0].metadata)

## Topic 9: Embeddings

**What:** An embedding model turns text into a list of numbers (a **vector**). Texts with similar *meaning* get vectors that are close to each other.

**Why HuggingFace and not Groq?** Groq only offers chat/LLM models, **no embedding model**. So we use a free model that runs on your own computer:
- `sentence-transformers/all-MiniLM-L6-v2` → small, fast, gives **384** numbers per text.
- The first run downloads the model (~90 MB) once.

`normalize_embeddings=True` makes every vector length = 1. This makes cosine similarity and dot product give the same result (we prove this in Topic 15).

In [ ]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

test_vector = embeddings.embed_query("What is the Transformer?")

print(f"Embedding dimensions: {len(test_vector)}")
print(f"First 10 values: {test_vector[:10]}")

## Topic 10: Vector Store (Chroma)

**What:** A vector store is a database that saves each chunk together with its embedding, and can quickly find the chunks closest to a query vector.

**Key settings:**
- `persist_directory` → saves the database on disk, so we don't have to embed again every time.
- `collection_name` → name of the table inside Chroma.
- `"hnsw": {"space": "cosine"}` → use **cosine** distance. HNSW is the fast search index Chroma uses.

**`REBUILD_INDEX` flag:**
- `True` → delete the old database and build a new one.
- `False` → just load the saved database (faster).

> **Important:** a database built with one embedding model can't be searched with another (OpenAI vectors had 1536 numbers, MiniLM has 384). Always keep the same embedding model for building and searching.

> **This is the only vector store of the notebook.** Every topic after this one uses `vector_store` (dense search) or `chunks` (BM25). The only extra collection is the small *child-chunk* collection that the Parent Document Retriever needs (Topic 21), and it lives in the **same** Chroma folder.

In [ ]:
PERSIST_DIRECTORY = DATA_DIR / "chroma_attention_retriever_hf"
COLLECTION_NAME = "attention_retriever_demo_hf"

# Set to False after the first run to reuse the saved index.
REBUILD_INDEX = True

if REBUILD_INDEX or not PERSIST_DIRECTORY.exists():
    shutil.rmtree(PERSIST_DIRECTORY, ignore_errors=True)

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(PERSIST_DIRECTORY),
        collection_configuration={"hnsw": {"space": "cosine"}},
    )
    print("New vector store created.")
else:
    vector_store = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=str(PERSIST_DIRECTORY),
    )
    print("Existing vector store loaded.")

print(f"Stored chunks: {len(vector_store.get()['ids'])}")
print(f"Saved at: {PERSIST_DIRECTORY}")

---
# Part C — Retrievers

The retriever is the **search** step of RAG. In this part we learn different ways to search the same vector store.

## Topic 11: What is a Retriever?

**Vector store vs Retriever:**
- A **vector store** stores and searches vectors.
- A **retriever** is a simple interface: *give it a question → it returns a list of relevant `Document`s*. You call it with `.invoke(query)`.

Any vector store can become a retriever with `vector_store.as_retriever(...)`. Because every retriever has the same `.invoke()` interface, you can swap search strategies without changing the rest of your RAG code.

**The 3 `search_type` options:**
| `search_type` | Idea |
|---|---|
| `"similarity"` | Return the top-k most similar chunks |
| `"mmr"` | Return relevant **and** diverse chunks |
| `"similarity_score_threshold"` | Return only chunks above a minimum score |

**Custom retrievers:** LangChain gives a `BaseRetriever` class. To build your own retriever, inherit from it and write your logic inside `_get_relevant_documents()`. Then it works with `.invoke()` like every other retriever — and plugs into chains and LangGraph. We build many custom retrievers in this notebook (Parent Document, Self-Query, MultiQuery, RAG Fusion, HyDE, Compression).

## Topic 12: Similarity Search Retriever

**What:** The basic retriever. It converts the query into a vector and returns the **k** closest chunks.

- `k=4` → return 4 chunks.

**Weakness:** the top results can be **near-duplicates** (e.g. 4 chunks saying the same thing), so you waste context space.

In [ ]:
similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

query = "What is scaled dot-product attention?"
similarity_documents = similarity_retriever.invoke(query)

display_documents(similarity_documents, title="Similarity Search Results")

## Topic 13: MMR Retriever (Maximal Marginal Relevance)

**Problem it solves:** similarity search may return repeated information.

**How MMR works:**
1. First fetch a bigger pool of candidates (`fetch_k=20`).
2. Then pick `k=4` one by one. Each new pick must be **relevant to the query** but **different from the chunks already picked**.

**`lambda_mult` (0 to 1):**
| Value | Behaviour |
|---|---|
| close to **1** | more relevance (behaves like normal similarity) |
| close to **0** | more diversity |
| **0.5** | balance of both |

**Use MMR when** you want wide coverage of a topic (e.g. "How is the Transformer designed?" touches the encoder, decoder, attention, feed-forward layers, embeddings...).

In [ ]:
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    },
)

query = "How is the Transformer model architecture designed?"
mmr_documents = mmr_retriever.invoke(query)

display_documents(mmr_documents, title="MMR Results")

### Similarity vs MMR — side by side
Look at the pages: similarity results usually come from the same area, MMR results are spread across more pages.

In [ ]:
query = "How is the Transformer model architecture designed?"

show_ids(similarity_retriever.invoke(query), "Similarity Search results:")
show_ids(mmr_retriever.invoke(query), "MMR results:")

## Topic 14: Relevance Scores & Score-Threshold Retriever

**Relevance score:** a number between **0 and 1** telling how relevant a chunk is (higher = better). With cosine space, `relevance = 1 − cosine distance`.

**Step 1 — look at the scores first** with `similarity_search_with_relevance_scores()`. This tells you what a "good" score looks like for your embedding model.

**Step 2 — score-threshold retriever:** returns only chunks whose score is **above** `score_threshold` (max `k` chunks). If nothing is good enough, it returns **nothing**, which is useful to avoid sending junk to the LLM.

> The right threshold depends on the embedding model. MiniLM scores are different from OpenAI scores, so check Step 1 and adjust the threshold. We use the same idea again in Topic 33 (`EmbeddingsFilter`).

In [ ]:
query = "How does multi-head attention work?"

scored_results = vector_store.similarity_search_with_relevance_scores(query=query, k=5)

for rank, (document, relevance_score) in enumerate(scored_results, start=1):
    print("=" * 90)
    print(f"Rank: {rank} | Relevance score: {relevance_score:.4f} | Page: {document.metadata.get('paper_page')} | Section: {document.metadata.get('section')}")
    print(document.page_content[:300])

In [ ]:
# Adjust this value after looking at the scores above.
SCORE_THRESHOLD = 0.5

threshold_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 10,
        "score_threshold": SCORE_THRESHOLD,
    },
)

threshold_documents = threshold_retriever.invoke(query)

display_documents(threshold_documents, title=f"Score-Threshold Results (score >= {SCORE_THRESHOLD})")

## Topic 15: Distance Metrics (Cosine, Euclidean, Dot Product)

These are the 3 common ways to measure how close two vectors are:

| Metric | Formula idea | Better when | Looks at |
|---|---|---|---|
| **Cosine similarity** | angle between vectors | **higher** | direction only (meaning) |
| **Euclidean distance** | straight-line distance | **lower** | direction + length |
| **Dot product** | multiply & add | **higher** | direction + length |

**Key fact:** when vectors are **normalized** (length = 1):
- dot product = cosine similarity
- euclidean distance gives the **same ranking** as cosine

So for normalized embeddings, all three metrics rank the chunks in the same order. We will prove it below.

In [ ]:
metric_query = "Why is self-attention better than recurrent layers?"

candidate_documents = vector_store.similarity_search(metric_query, k=6)
candidate_texts = [document.page_content for document in candidate_documents]

query_vector = np.asarray(embeddings.embed_query(metric_query), dtype=np.float64)
document_vectors = np.asarray(embeddings.embed_documents(candidate_texts), dtype=np.float64)

print("Query-vector shape:", query_vector.shape)
print("Document-vectors shape:", document_vectors.shape)

In [ ]:
def cosine_similarity(vector_a: np.ndarray, vector_b: np.ndarray) -> float:
    denominator = np.linalg.norm(vector_a) * np.linalg.norm(vector_b)
    if denominator == 0:
        return 0.0
    return float(np.dot(vector_a, vector_b) / denominator)


def euclidean_distance(vector_a: np.ndarray, vector_b: np.ndarray) -> float:
    return float(np.linalg.norm(vector_a - vector_b))


def dot_product(vector_a: np.ndarray, vector_b: np.ndarray) -> float:
    return float(np.dot(vector_a, vector_b))


metric_rows = []

for document, document_vector in zip(candidate_documents, document_vectors):
    metric_rows.append(
        {
            "chunk_id": document.metadata.get("chunk_id"),
            "section": document.metadata.get("section"),
            "cosine_similarity": cosine_similarity(query_vector, document_vector),
            "euclidean_distance": euclidean_distance(query_vector, document_vector),
            "dot_product": dot_product(query_vector, document_vector),
            "preview": document.page_content[:80].replace("\n", " "),
        }
    )

metric_table = pd.DataFrame(metric_rows)
metric_table

### Ranking by each metric
Cosine and dot product are sorted **high → low**, euclidean is sorted **low → high**. The chunk order comes out the same in all three.

In [ ]:
print("By cosine (high first):     ", metric_table.sort_values("cosine_similarity", ascending=False)["chunk_id"].tolist())
print("By euclidean (low first):   ", metric_table.sort_values("euclidean_distance", ascending=True)["chunk_id"].tolist())
print("By dot product (high first):", metric_table.sort_values("dot_product", ascending=False)["chunk_id"].tolist())

### Proof: normalized dot product = cosine similarity

In [ ]:
normalized_query_vector = query_vector / np.linalg.norm(query_vector)
normalized_document_vectors = document_vectors / np.linalg.norm(document_vectors, axis=1, keepdims=True)

normalized_dot_scores = normalized_document_vectors @ normalized_query_vector
cosine_scores = np.asarray([cosine_similarity(query_vector, v) for v in document_vectors])

print("Cosine scores:              ", cosine_scores)
print("Dot product after normalize:", normalized_dot_scores)
print("Approximately equal?", np.allclose(cosine_scores, normalized_dot_scores, atol=1e-8))

## Topic 16: Metadata Filtering

**What:** Search **only** among chunks whose metadata matches a condition. This is where the metadata from Topic 7 pays off.

**Simple filter:** `{"section": "model_architecture"}` → only model-architecture chunks.

**Chroma filter operators:**
| Operator | Meaning |
|---|---|
| `$eq` / `$ne` | equal / not equal |
| `$gt`, `$gte`, `$lt`, `$lte` | greater / less than |
| `$in` / `$nin` | value is in / not in a list |
| `$and` / `$or` | combine several conditions |

> With more than one condition, Chroma needs an explicit `$and` (or `$or`).

In [ ]:
architecture_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {"section": "model_architecture"},
    },
)

architecture_documents = architecture_retriever.invoke("How does the decoder use attention?")
display_documents(architecture_documents, title="Filter: section = model_architecture")

# Check: every result must be from model_architecture
for document in architecture_documents:
    assert document.metadata["section"] == "model_architecture"
print("\nAll returned documents are from the model_architecture section.")

### Multiple conditions with `$and`

In [ ]:
filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {
            "$and": [
                {"section": {"$eq": "training"}},
                {"year": {"$eq": 2017}},
                {"organization": {"$eq": "Google"}},
            ]
        },
    },
)

filtered_documents = filtered_retriever.invoke("Which optimizer and learning rate schedule were used?")
display_documents(filtered_documents, title="Filter: training AND 2017 AND Google")

## Topic 17: Pre-filtering vs Post-filtering

| | **Pre-filtering** | **Post-filtering** |
|---|---|---|
| When is the filter applied? | **Before** the vector search | **After** the vector search |
| How | the database only searches matching chunks | search everything, then remove non-matching results in Python |
| Number of results | always gets `k` matching results (if they exist) | may end up with **fewer** than you wanted |
| Speed | faster on big datasets | wastes work on chunks that get thrown away |
| When to use | filter is known in advance (section, user, date) | filter logic is complex and the database can't express it |

The `filter` we passed to Chroma above **is pre-filtering**.

In [ ]:
# PRE-FILTERING: filter goes into the search itself
pre_filter = {
    "$and": [
        {"section": {"$eq": "results"}},
        {"year": {"$eq": 2017}},
    ]
}

pre_filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4, "filter": pre_filter},
)

pre_filtered_documents = pre_filtered_retriever.invoke("What BLEU scores did the Transformer achieve?")
display_documents(pre_filtered_documents, title="Pre-filtering Results")

In [ ]:
# POST-FILTERING: search first (no filter), then filter in Python
unfiltered_candidates = vector_store.similarity_search(query="What BLEU scores did the Transformer achieve?", k=15)

post_filtered_documents = [
    document
    for document in unfiltered_candidates
    if document.metadata.get("section") == "results"
    and document.metadata.get("year") == 2017
]

print("Documents before post-filtering:", len(unfiltered_candidates))
print("Documents after post-filtering: ", len(post_filtered_documents))

display_documents(post_filtered_documents, title="Post-filtering Results", max_documents=5)

## Topic 18: Sparse Retrieval (BM25)

**What:** Classic **keyword search**. It does not understand meaning; it matches the actual words.

**Why "sparse":** each text is represented by a huge vector of all vocabulary words, where most values are 0 (only the words present are non-zero).

**How BM25 scores a chunk:**
- **Term frequency** — the query word appears many times in the chunk → higher score (but with diminishing returns).
- **Rare words matter more (IDF)** — a rare word like "BLEU" or "WMT" is more valuable than a common word like "the".
- **Length normalization** — long chunks don't win just because they are long.

**Strong at:** exact terms, abbreviations, names, codes, numbers (`BLEU`, `WMT 2014`, `d_model`, `28.4`).
**Weak at:** synonyms. "word order" will not match "positional encoding".

BM25 does **not** need embeddings; it builds its index directly from the same `chunks` we created in Topic 8.

In [ ]:
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 4  # number of results

sparse_query = "BLEU WMT 2014 English-to-German"
sparse_documents = bm25_retriever.invoke(sparse_query)

display_documents(sparse_documents, title="Sparse Retrieval: BM25 Results")

## Topic 19: Dense Retrieval

**What:** **Meaning-based search** using embeddings (what we did in Topics 12–14).

**Why "dense":** every text is a short vector (384 numbers) where almost every value is non-zero.

**Strong at:** understanding paraphrases and synonyms. The query below never says "BLEU" or "WMT", but dense search can still find the translation-results chunks because the *meaning* matches.
**Weak at:** exact rare keywords, IDs and numbers.

In [ ]:
dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

dense_query = "How well did the model translate from English to German compared to earlier systems?"
dense_documents = dense_retriever.invoke(dense_query)

display_documents(dense_documents, title="Dense Retrieval: Vector Search Results")

### Sparse vs Dense — same query
| | Sparse (BM25) | Dense (Embeddings) |
|---|---|---|
| Matches | exact words | meaning |
| Synonyms | ❌ | ✅ |
| Exact codes / abbreviations | ✅ | ⚠️ sometimes misses |
| Needs embedding model | No | Yes |

Each one finds some chunks the other misses. That is why we combine them next.

In [ ]:
comparison_query = "What BLEU score did the big Transformer get on English-to-German translation?"

show_ids(bm25_retriever.invoke(comparison_query), "SPARSE (BM25) RESULTS")
show_ids(dense_retriever.invoke(comparison_query), "DENSE RESULTS")

## Topic 20: Hybrid Retrieval / Ensemble Retriever (Weighted RRF)

**What:** Run **two or more retrievers**, then merge their result lists into one ranking. With BM25 + dense you get exact-keyword power **and** meaning power.

**How the merge works — Reciprocal Rank Fusion (RRF):**
RRF only looks at the **rank position** of each chunk, not the raw scores (BM25 scores and cosine scores are on different scales, so they can't be added directly).

$$\text{score}(d) = \sum_{\text{each retriever}} \frac{\text{weight}}{c + \text{rank}(d)}$$

- `rank` = position of the chunk in that retriever's list (1, 2, 3...)
- `c` = 60 (a constant that stops the top rank from dominating)
- `weight` = how much you trust each retriever

A chunk that appears high in **both** lists gets the biggest final score.

`EnsembleRetriever` does all of this for us. `weights=[0.5, 0.5]` → equal trust in BM25 and dense.

```
                    ┌── BM25 Retriever ─────┐
User query ─────────┤                       ├── Weighted RRF ──► Combined ranking
                    └── Dense Retriever ────┘
```

In [ ]:
# Give each retriever a bigger pool so the fusion has more to work with
bm25_retriever.k = 8

dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8},
)

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5],  # [BM25 weight, Dense weight]
)

hybrid_query = "Transformer big model BLEU score EN-DE 28.4 training cost"
hybrid_documents = hybrid_retriever.invoke(hybrid_query)

display_documents(hybrid_documents, title="Hybrid Retrieval: BM25 + Dense + RRF", max_documents=6)

### Comparing all three on one query

In [ ]:
test_query = "How expensive was it to train the Transformer compared to other models?"

show_ids(bm25_retriever.invoke(test_query)[:4], "1. Sparse Retrieval")
show_ids(dense_retriever.invoke(test_query)[:4], "2. Dense Retrieval")
show_ids(hybrid_retriever.invoke(test_query)[:4], "3. Hybrid Retrieval")

### Ensemble is not only BM25 + Dense
`EnsembleRetriever` can fuse **any** retrievers. A second common combo is **Similarity + MMR** on the same vector store:
- similarity brings the most relevant chunks,
- MMR brings diverse chunks,
- RRF puts chunks that both agree on at the top.

(This was the "RAG Fusion with Ensemble Retriever" idea in the class notebook. RAG Fusion with **LLM-generated queries** comes in Topic 28.)

In [ ]:
similarity_mmr_ensemble = EnsembleRetriever(
    retrievers=[
        vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4}),
        vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.5}),
    ],
    weights=[0.5, 0.5],
)

show_ids(similarity_mmr_ensemble.invoke("How does the Transformer handle long-range dependencies?"), "Similarity + MMR ensemble:")

## Topic 21: Parent Document Retriever

**The chunk-size dilemma:**
| Small chunks | Big chunks |
|---|---|
| ✅ precise embeddings — search finds the exact spot | ❌ one vector for many ideas — search gets blurry |
| ❌ too little context for the LLM to answer | ✅ enough context for a full answer |

**Idea:** use **both**.
1. Split pages into **big parent chunks** (1500 chars) → store them in a **docstore** (a key-value store).
2. Split every parent into **small child chunks** (400 chars) → embed them in a vector store. Each child remembers its parent's id.
3. At query time: search the **children** (precise), then return their **parents** (more context).

```
query ──► search small child chunks ──► collect parent ids ──► return big parent chunks
```

**Why a separate collection here?** Our main collection holds 1000-char chunks. The child chunks are a different, smaller index, so they need their own collection. We keep it in the **same Chroma folder**, built from the **same `pages`** (with the same metadata), and we reset it on every run so no duplicates pile up.

**Two kinds of docstore:**
| Docstore | Where parents live | Survives restart? |
|---|---|---|
| `InMemoryStore` | a Python dict in RAM | ❌ no |
| `LocalFileStore` + `create_kv_docstore` | JSON files on disk | ✅ yes |

`LocalFileStore` can only store raw bytes, so `create_kv_docstore()` wraps it with a serializer that turns `Document` objects into bytes and back.

In [ ]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

# --- Part 1: InMemoryStore ---
parent_store_memory = InMemoryStore()

child_vector_store_memory = Chroma(
    collection_name=f"{COLLECTION_NAME}_parent_children",
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIRECTORY),
    collection_configuration={"hnsw": {"space": "cosine"}},
)
child_vector_store_memory.reset_collection()  # start clean on every run (no duplicate children)

parent_retriever_memory = ParentDocumentRetriever(
    vectorstore=child_vector_store_memory,
    docstore=parent_store_memory,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_kwargs={"k": 4},  # how many child chunks to search for
)
parent_retriever_memory.add_documents(pages)

parent_count = len(list(parent_store_memory.yield_keys()))
child_count = len(child_vector_store_memory.get()["ids"])
print(f"InMemoryStore  |  Parent chunks: {parent_count}  |  Child chunks: {child_count}")

In [ ]:
# Similarity search hits small child chunks, but the retriever returns the bigger parent chunks
parent_query = "How does the decoder prevent positions from attending to future positions?"

child_hits = child_vector_store_memory.similarity_search(parent_query, k=4)
print("Child chunk sizes found by search:", [len(d.page_content) for d in child_hits])

parent_documents = parent_retriever_memory.invoke(parent_query)
print("Parent chunk sizes returned:     ", [len(d.page_content) for d in parent_documents])

display_documents(parent_documents, title="Parent Document Retriever (InMemoryStore)", max_characters=900)

In [ ]:
# --- Part 2: LocalFileStore (parents are saved on disk) ---
PARENT_STORE_DIRECTORY = DATA_DIR / "attention_parent_docstore"
shutil.rmtree(PARENT_STORE_DIRECTORY, ignore_errors=True)  # start clean on every run

file_store = LocalFileStore(str(PARENT_STORE_DIRECTORY))
parent_store_disk = create_kv_docstore(file_store)

child_vector_store_disk = Chroma(
    collection_name=f"{COLLECTION_NAME}_parent_children_disk",
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIRECTORY),
    collection_configuration={"hnsw": {"space": "cosine"}},
)
child_vector_store_disk.reset_collection()

parent_retriever_disk = ParentDocumentRetriever(
    vectorstore=child_vector_store_disk,
    docstore=parent_store_disk,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_kwargs={"k": 4},
)
parent_retriever_disk.add_documents(pages)

print(f"LocalFileStore  |  parent files on disk: {len(list(file_store.yield_keys()))}  |  child chunks: {len(child_vector_store_disk.get()['ids'])}")

# Same query -> same parents, proving both stores behave the same
show_ids(parent_retriever_memory.invoke(parent_query), "InMemoryStore results:")
show_ids(parent_retriever_disk.invoke(parent_query), "LocalFileStore results:")

## Topic 22: Custom Parent Document Retriever

Same idea as Topic 21, written by hand so you can see every step.

| Step | What happens |
|---|---|
| 1 | Split pages into **parent** chunks |
| 2 | Give every parent a unique **id** (`uuid`) |
| 3 | Split each parent into **child** chunks and tag every child with its parent id → add children to the vector store |
| 4 | Save the parent in the docstore: `docstore.mset([(id, parent)])` (like `dict[id] = parent`) |
| 5 | At query time: similarity search over **children** |
| 6 | Collect the **unique** parent ids (order kept) and fetch parents: `docstore.mget(ids)` (like `dict[id]`) |

In [ ]:
class CustomParentDocumentRetriever(BaseRetriever):
    # Index small child chunks for search, but return the larger parent chunks.

    vectorstore: Any          # Chroma: stores child chunks for similarity search
    docstore: Any             # InMemoryStore: stores parent chunks keyed by id
    child_splitter: Any
    parent_splitter: Any
    id_key: str = "parent_doc_id"
    k: int = 4

    def add_documents(self, docs: List[Document]) -> None:
        parent_chunks = self.parent_splitter.split_documents(docs)              # Step 1
        for parent in parent_chunks:
            parent_id = str(uuid.uuid4())                                       # Step 2
            children = self.child_splitter.split_documents([parent])           # Step 3
            for child in children:
                child.metadata[self.id_key] = parent_id
            self.vectorstore.add_documents(children)
            self.docstore.mset([(parent_id, parent)])                           # Step 4

    def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun) -> List[Document]:
        child_docs = self.vectorstore.similarity_search(query, k=self.k)        # Step 5
        parent_ids = list(dict.fromkeys(                                        # Step 6 (unique, order kept)
            doc.metadata[self.id_key] for doc in child_docs if self.id_key in doc.metadata
        ))
        return [doc for doc in self.docstore.mget(parent_ids) if doc is not None]

In [ ]:
custom_child_vector_store = Chroma(
    collection_name=f"{COLLECTION_NAME}_custom_parent_children",
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIRECTORY),
    collection_configuration={"hnsw": {"space": "cosine"}},
)
custom_child_vector_store.reset_collection()

custom_parent_docstore = InMemoryStore()

custom_parent_retriever = CustomParentDocumentRetriever(
    vectorstore=custom_child_vector_store,
    docstore=custom_parent_docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)
custom_parent_retriever.add_documents(pages)

print(f"Parent chunks: {len(list(custom_parent_docstore.yield_keys()))}  |  Child chunks: {len(custom_child_vector_store.get()['ids'])}")

custom_parent_documents = custom_parent_retriever.invoke(parent_query)
display_documents(custom_parent_documents, title="Custom Parent Document Retriever", max_characters=900)

## Topic 23: Self-Query Retriever

**Problem:** users mix *meaning* and *filters* in one sentence:
> *"What does the **training section** say about regularization?"*

Plain vector search ignores the "training section" part. Metadata filtering (Topic 16) works, but **you** had to write the filter by hand.

**Idea:** let the **LLM** read the question and split it into:
1. a **semantic query** → `"regularization"` (used for vector search)
2. a **metadata filter** → `section == "training"` (used as a Chroma filter)

```
"What does the training section say about regularization?"
        │  LLM (knows which metadata fields exist)
        ▼
query = "regularization"     filter = eq("section", "training")
        │                              │  ChromaTranslator
        ▼                              ▼
          vector_store.similarity_search(query, filter={"section": {"$eq": "training"}})
```

**What the LLM needs to know:**
- `document_contents` → a one-line description of what the chunks are.
- `metadata_field_info` → list of `AttributeInfo(name, description, type)` for every field it may filter on. Listing the **allowed values** (our section names) helps a lot.
- `ChromaTranslator()` → converts the LLM's generic filter into Chroma's `$eq / $gte / $and` syntax.
- `enable_limit=True` → the LLM may also set *how many* results to return ("give me 2 chunks…").

> This uses the **same `vector_store`** and the metadata we added in Topic 7.

In [ ]:
SECTION_NAMES = [
    "front_matter", "introduction", "model_architecture", "why_self_attention",
    "training", "results", "conclusion", "references", "attention_visualizations",
]

metadata_field_info = [
    AttributeInfo(
        name="section",
        description="The section of the paper. One of: " + ", ".join(SECTION_NAMES),
        type="string",
    ),
    AttributeInfo(
        name="paper_page",
        description="The page number of the paper, from 1 to 15",
        type="integer",
    ),
    AttributeInfo(
        name="year",
        description="The year the paper was published",
        type="integer",
    ),
]

document_content_description = "Chunks of the research paper 'Attention Is All You Need', which introduces the Transformer"

self_query_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vector_store,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    structured_query_translator=ChromaTranslator(),
    enable_limit=True,
)

In [ ]:
# Baseline: plain vector search ignores the "training section" part of the question
show_ids(dense_retriever.invoke("What does the training section say about regularization?")[:4], "Plain dense search:")

# Self-query: LLM extracts query="regularization", filter: section == "training"
self_query_documents = self_query_retriever.invoke("What does the training section say about regularization?")
display_documents(self_query_documents, title="Self-Query: section filter")

In [ ]:
# Number filter: the LLM should create  paper_page >= 8  (and may set a limit of 2)
self_query_documents = self_query_retriever.invoke("Give me 2 chunks about BLEU scores from page 8 or later")
display_documents(self_query_documents, title="Self-Query: page filter + limit")

# No filter in the question -> the LLM uses only the semantic query
self_query_documents = self_query_retriever.invoke("How are positional encodings computed?")
display_documents(self_query_documents, title="Self-Query: no filter (pure semantic)")

## Topic 24: Custom Self-Query Retriever

Same idea, built by hand with **structured output** so you can see exactly what the LLM produces.

```
query ──► LLM + Pydantic schema ──► SelfQuerySchema(query=..., filters=[...]) ──► Chroma filter dict ──► similarity_search
```

**Two Pydantic models:**
- `MetadataFilter` → one condition: `field`, `value`, `operator` (`eq, ne, gt, gte, lt, lte`)
- `SelfQuerySchema` → the semantic `query` + an optional list of filters

`llm.with_structured_output(SelfQuerySchema)` forces the LLM to return a `SelfQuerySchema` object instead of free text.

**Small improvement over the class version:** LLMs sometimes return numbers as text (`"8"` instead of `8`). Chroma's `$gte` needs a real number, so we convert integer fields before searching. We also ignore fields that don't exist.

In [ ]:
class MetadataFilter(BaseModel):
    field: str = Field(description="The metadata field name to filter on: section, paper_page or year")
    value: str | int | float = Field(description="The value to filter by")
    operator: str = Field(default="eq", description="Comparison operator: eq, ne, gt, gte, lt, lte")


class SelfQuerySchema(BaseModel):
    query: str = Field(description="The semantic search query extracted from the user's query")
    filters: Optional[List[MetadataFilter]] = Field(
        default=None,
        description="Metadata filters extracted from the query. None if no filters apply.",
    )


self_query_system_prompt = (
    "You are a query parser for chunks of the research paper 'Attention Is All You Need'. "
    "Parse the user's natural language query into:\n"
    "1. A semantic search query (the conceptual meaning to search for)\n"
    "2. Optional metadata filters on these fields:\n"
    "   - section (string): one of " + ", ".join(SECTION_NAMES) + "\n"
    "   - paper_page (integer): page number from 1 to 15\n"
    "   - year (integer): publication year\n\n"
    "Supported operators: eq, ne, gt, gte, lt, lte\n"
    "Only add filters when the user explicitly specifies metadata constraints."
)

self_query_prompt = ChatPromptTemplate.from_messages([
    ("system", self_query_system_prompt),
    ("human", "query: {query}"),
])

self_query_chain = self_query_prompt | llm.with_structured_output(SelfQuerySchema)

parsed = self_query_chain.invoke({"query": "What does the training section say about regularization?"})
parsed

In [ ]:
class CustomSelfQueryRetriever(BaseRetriever):
    # Use an LLM to parse a natural language query into a semantic query + Chroma filters.

    vectorstore: Any
    query_chain: Any
    k: int = 4

    allowed_fields: tuple = ("section", "paper_page", "year")
    integer_fields: tuple = ("paper_page", "year")

    def _build_chroma_filter(self, filters: List[MetadataFilter]) -> Optional[dict]:
        op_map = {"eq": "$eq", "ne": "$ne", "gt": "$gt", "gte": "$gte", "lt": "$lt", "lte": "$lte"}
        conditions = []

        for f in filters:
            if f.field not in self.allowed_fields:
                continue  # ignore fields that don't exist in our metadata
            value = f.value
            if f.field in self.integer_fields:
                value = int(float(value))  # "8" -> 8
            conditions.append({f.field: {op_map.get(f.operator, "$eq"): value}})

        if not conditions:
            return None
        if len(conditions) == 1:
            return conditions[0]                      # {"section": {"$eq": "training"}}
        return {"$and": conditions}                   # {"$and": [{...}, {...}]}

    def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun) -> List[Document]:
        parsed: SelfQuerySchema = self.query_chain.invoke({"query": query})          # Step 1: parse
        chroma_filter = self._build_chroma_filter(parsed.filters or [])               # Step 2: translate
        print(f"Parsed query: '{parsed.query}'  |  Chroma filter: {chroma_filter}")
        return self.vectorstore.similarity_search(parsed.query, k=self.k, filter=chroma_filter)  # Step 3: search


custom_self_query_retriever = CustomSelfQueryRetriever(vectorstore=vector_store, query_chain=self_query_chain)

In [ ]:
print("=== Query 1: section filter ===")
show_ids(custom_self_query_retriever.invoke("What does the training section say about regularization?"), "Results:")

print("\n=== Query 2: page range filter ===")
show_ids(custom_self_query_retriever.invoke("Explain positional encoding using only pages 5 to 6"), "Results:")

print("\n=== Query 3: pure semantic, no filter expected ===")
show_ids(custom_self_query_retriever.invoke("Why did the authors use self-attention instead of convolutions?"), "Results:")

---
# Part D — Query Transformation (before retrieval)

Even the best retriever fails if the **question** is bad: too vague, a follow-up with "that", using different words than the paper, or asking many things at once. These techniques use the LLM to **improve the query before searching**.

| Problem with the query | Technique |
|---|---|
| follow-up question ("and how is it different?") | Query rewriting (25) |
| user words ≠ paper words | Query expansion / MultiQuery (26, 27) |
| want a better *ranking* from many queries | RAG Fusion (28) |
| several questions in one | Query decomposition (29) |
| short question vs long technical text | HyDE (30, 31) |

## Topic 25: Query Rewriting

**Problem:** In a chat, users ask follow-up questions like *"And how is the decoder different from that?"*. The word **"that"** means nothing to a retriever, so search fails.

**Solution:** Give the LLM the conversation history and ask it to **rewrite the question into a complete, standalone search query**.

```
"And how is the decoder different from that?"
            ↓  (LLM + chat history)
"How does the Transformer decoder differ from the encoder stack of six identical layers?"
```

The chain is: `prompt | llm | StrOutputParser()` → the parser returns plain text instead of a message object.

In [ ]:
query_rewriting_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            '''You rewrite conversational questions into clear, standalone search queries.

Rules:
1. Do not answer the question.
2. Preserve important entities, dates, and technical terms.
3. Resolve pronouns using the conversation history.
4. Return only one rewritten query.''',
        ),
        (
            "human",
            '''Conversation history:
{chat_history}

Current query:
{query}''',
        ),
    ]
)

query_rewriting_chain = query_rewriting_prompt | llm | StrOutputParser()

In [ ]:
chat_history = '''
User: What does the encoder of the Transformer consist of?
Assistant: A stack of N=6 identical layers, each with a multi-head self-attention sub-layer and a position-wise feed-forward network.
'''

original_query = "And how is the decoder different from that?"

rewritten_query = query_rewriting_chain.invoke(
    {"chat_history": chat_history, "query": original_query}
).strip()

print("Original query: ", original_query)
print("Rewritten query:", rewritten_query)

show_ids(hybrid_retriever.invoke(original_query)[:4], "Retrieved with the ORIGINAL query:")
rewritten_query_documents = hybrid_retriever.invoke(rewritten_query)
display_documents(rewritten_query_documents, title="Documents Retrieved Using the Rewritten Query", max_documents=4)

## Topic 26: Query Expansion (Custom MultiQuery Retriever)

**Problem:** The user's words may not match the words used in the paper. A user asks *"How does the model know the order of words?"* — the paper calls it **positional encoding** with **sine and cosine functions**.

**Solution:** Ask the LLM to generate **several versions of the same question** (synonyms, abbreviations, technical terms). Search with **all** of them, combine the results and remove duplicates. This increases **recall** (finding more of the relevant chunks).

**Structured output:** `llm.with_structured_output(ExpandedQueryOutput)` forces the LLM to return a clean Python object (a list of queries) instead of free text, so we don't have to parse it ourselves.

```
original query ──► LLM ──► 4 alternative queries ──► retrieve for each (+ original) ──► merge ──► remove duplicates
```

We wrap the whole thing in a **custom retriever** (`BaseRetriever`) so it can be used with `.invoke()` like any other retriever — this is exactly what LangChain's built-in `MultiQueryRetriever` does inside (next topic).

In [ ]:
class ExpandedQueryOutput(BaseModel):
    queries: List[str] = Field(
        description="Four alternative search queries expressing the same information need using different wording."
    )


query_expansion_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            '''Generate four alternative search queries for the user's query.

Use:
- synonyms,
- related technical terms,
- abbreviations where appropriate,
- alternative wording.

Do not answer the query.
Each query must preserve the original intent.''',
        ),
        ("human", "Original query: {query}"),
    ]
)

query_expansion_chain = query_expansion_prompt | llm.with_structured_output(ExpandedQueryOutput)


class CustomMultiQueryRetriever(BaseRetriever):
    # Generate alternative phrasings with an LLM, retrieve for each, return the unique union.

    base_retriever: Any
    query_chain: Any
    include_original: bool = True
    last_queries: List[str] = []

    def _generate_queries(self, query: str) -> List[str]:
        generated = self.query_chain.invoke({"query": query}).queries
        return [query, *generated] if self.include_original else generated

    def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun) -> List[Document]:
        queries = self._generate_queries(query)                     # Step 1: expand
        self.last_queries = queries

        all_documents = []
        for q in queries:                                           # Step 2: retrieve for every query
            all_documents.extend(self.base_retriever.invoke(q))

        return deduplicate_documents(all_documents)                 # Step 3: unique union


custom_multi_query_retriever = CustomMultiQueryRetriever(
    base_retriever=vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3}),
    query_chain=query_expansion_chain,
)


In [ ]:
expansion_query = "How does the model know the order of words?"

show_ids(vector_store.as_retriever(search_kwargs={"k": 3}).invoke(expansion_query), "Single-query search (k=3):")

expanded_query_documents = custom_multi_query_retriever.invoke(expansion_query)

print("\nGenerated search queries:")
for number, q in enumerate(custom_multi_query_retriever.last_queries, start=1):
    print(f"{number}. {q}")

display_documents(expanded_query_documents, title="Query Expansion: Combined Unique Documents", max_documents=10)

## Topic 27: MultiQueryRetriever (Built-in Query Expansion)

LangChain already has query expansion built in. `MultiQueryRetriever`:
1. asks the LLM for several (default 3) versions of the question,
2. runs the base retriever for each,
3. returns the **unique union** of all results.

`include_original=True` → also searches with the user's original question, so documents that match the original wording are never missed.

It is the same idea as Topic 26, but in just a few lines. Use the **built-in** one for quick work, the **custom** one when you want control over the prompt, the number of queries or to see the generated queries.

In [ ]:
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3}),
    llm=llm,
    include_original=True,
)

multi_query_documents = multi_query_retriever.invoke("Why did the authors remove recurrence from the model?")
display_documents(multi_query_documents, title="Built-in MultiQueryRetriever Results", max_documents=10)

## Topic 28: RAG Fusion (Multi-Query + Reciprocal Rank Fusion)

**MultiQuery vs RAG Fusion:**
| | MultiQuery (26–27) | RAG Fusion |
|---|---|---|
| Generate several queries | ✅ | ✅ |
| Retrieve for each | ✅ | ✅ |
| How results are merged | plain **union** (order = whichever query found it first) | **RRF ranking**: chunks found by many queries, near the top, win |
| Output | many docs, no real ranking | a **ranked** top-n list |

RRF is the same formula as the Ensemble retriever (Topic 20), but here the *lists* come from **different queries** instead of different retrievers:

$$\text{rrf}(d) = \sum_{\text{each query}} \frac{1}{60 + \text{rank}(d)}$$

```
query ──► LLM ──► sub-queries ──┬─► retrieve ──► ranked list 1 ─┐
                                ├─► retrieve ──► ranked list 2 ─┼─► RRF ──► top-n fused documents
                                └─► retrieve ──► ranked list 3 ─┘
```

The class notebook imported `RAGFusion` from a local `rag_fusion.py` file. Here the full logic is written in the notebook as a custom retriever, so nothing extra is needed. It reuses `query_expansion_chain` from Topic 26 to create the queries.

In [ ]:
def reciprocal_rank_fusion(ranked_lists: List[List[Document]], c: int = 60) -> List[Document]:
    # Merge several ranked lists of documents with RRF. Returns documents sorted by fused score.
    scores = {}
    documents_by_key = {}

    for ranked_documents in ranked_lists:
        for rank, document in enumerate(ranked_documents, start=1):
            key = document.metadata.get("chunk_id") or document.page_content
            documents_by_key.setdefault(key, document)
            scores[key] = scores.get(key, 0.0) + 1.0 / (c + rank)

    fused = []
    for key in sorted(scores, key=scores.get, reverse=True):
        document = documents_by_key[key]
        fused.append(Document(page_content=document.page_content, metadata={**document.metadata, "rrf_score": scores[key]}))
    return fused


class RAGFusionRetriever(BaseRetriever):
    # Generate sub-queries with an LLM, retrieve for each, fuse the ranked lists with RRF.

    base_retriever: Any
    query_chain: Any
    top_n: int = 5
    rrf_c: int = 60
    include_original: bool = True
    last_queries: List[str] = []

    def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun) -> List[Document]:
        generated = self.query_chain.invoke({"query": query}).queries          # Step 1: sub-queries
        queries = [query, *generated] if self.include_original else generated
        self.last_queries = queries

        ranked_lists = [self.base_retriever.invoke(q) for q in queries]       # Step 2: one ranked list per query
        return reciprocal_rank_fusion(ranked_lists, c=self.rrf_c)[: self.top_n]  # Step 3: fuse + keep top-n


rag_fusion_retriever = RAGFusionRetriever(
    base_retriever=vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5}),
    query_chain=query_expansion_chain,
    top_n=5,
)

In [ ]:
fusion_query = "How does the Transformer handle long-range dependencies?"

fused_documents = rag_fusion_retriever.invoke(fusion_query)

print("Queries used:")
for number, q in enumerate(rag_fusion_retriever.last_queries, start=1):
    print(f"{number}. {q}")

display_documents(fused_documents, title="RAG Fusion: RRF-ranked documents", max_documents=5)

## Topic 29: Query Decomposition

**Problem:** A complex question asks **several things at once**, e.g. *"Compare the encoder and the decoder, and explain how the model was regularized"*. One search can't cover all parts well.

**Solution:** The LLM **breaks the question into small, independent (atomic) sub-questions**. We search for each one separately and merge the evidence.

**Expansion vs Decomposition:**
- **Expansion** → *same* question, different words.
- **Decomposition** → *different* smaller questions that together answer the big one.

```
Complex query
     ↓
Atomic sub-queries
     ↓
Retrieve for every sub-query
     ↓
Merge evidence → Remove duplicates
```

(We use decomposition again inside an **agent** in Topic 46, where the agent decides by itself when a question needs to be split.)

In [ ]:
class DecomposedQueryOutput(BaseModel):
    sub_queries: List[str] = Field(
        description="Independent and atomic search queries required to answer the complete user question."
    )


query_decomposition_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            '''Break the user's complex question into independent, atomic search queries.

Rules:
1. Do not answer the question.
2. Generate only sub-queries needed to answer it.
3. Each sub-query must be understandable independently.
4. Preserve named entities, model names, and dates.
5. Generate between two and five sub-queries.''',
        ),
        ("human", "Complex query: {query}"),
    ]
)

query_decomposition_chain = query_decomposition_prompt | llm.with_structured_output(DecomposedQueryOutput)

In [ ]:
complex_query = "Compare the encoder and the decoder of the Transformer, and explain how the model was regularized during training."

sub_queries = query_decomposition_chain.invoke({"query": complex_query}).sub_queries

print("Original complex query:\n", complex_query)
print("\nGenerated sub-queries:")
for number, q in enumerate(sub_queries, start=1):
    print(f"{number}. {q}")

# Retrieve for each sub-query
decomposition_results = {q: hybrid_retriever.invoke(q) for q in sub_queries}

for sub_query, documents in decomposition_results.items():
    show_ids(documents[:4], f"Sub-query: {sub_query}")

In [ ]:
# Merge evidence from all sub-queries and remove duplicates
all_decomposition_documents = []
for documents in decomposition_results.values():
    all_decomposition_documents.extend(documents[:4])

all_decomposition_documents = deduplicate_documents(all_decomposition_documents)

display_documents(all_decomposition_documents, title="Combined Evidence from All Sub-Queries", max_documents=12)

## Topic 30: HyDE — Hypothetical Document Embeddings (Custom Retriever)

**Problem:** A short question and a long answer paragraph look different, so their embeddings may not be very close.
- Question: *"How did they let the model look at different kinds of information at the same time?"*
- Paper text: *"Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions... h = 8 parallel attention layers..."*

**Idea:** Ask the LLM to **write a fake (hypothetical) answer paragraph** first. A fake answer *looks like* the real paper text, so its embedding lands closer to the real chunks. Then we search with the fake answer's embedding.

> The fake passage may contain wrong facts — that is fine. It is **only used for searching**. The final answer must always come from the **real retrieved documents** (Part F).

```
User query
     ↓
LLM writes a hypothetical answer passage
     ↓
Embed the hypothetical passage
     ↓
Dense vector search (same vector_store)
     ↓
Real source documents
```

**Why this custom version and not the class `hyde.py` version?** Both do the same job; this one is fully visible in the notebook and saves the hypothetical passage (`last_hypothetical_document`) so you can inspect it.

In [ ]:
class HyDERetriever(BaseRetriever):
    # Generate a hypothetical passage, then retrieve REAL documents with it.

    hypothesis_chain: Any
    vector_retriever: Any
    last_hypothetical_document: str = ""

    def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun) -> List[Document]:
        # Step 1: LLM writes a fake answer passage
        hypothetical_document = self.hypothesis_chain.invoke({"query": query}).strip()
        self.last_hypothetical_document = hypothetical_document

        # Step 2: search the vector store with that passage
        return self.vector_retriever.invoke(
            hypothetical_document,
            config={"callbacks": run_manager.get_child()},
        )

In [ ]:
hyde_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            '''Write a concise research-paper passage that would directly answer the user's question.
Use technical terminology and named entities likely to appear in the source document.
Do not mention that the passage is hypothetical.
Do not add citations, headings, or commentary.
Return only the passage.''',
        ),
        ("human", "Question: {query}"),
    ]
)

hyde_hypothesis_chain = hyde_prompt | llm | StrOutputParser()

hyde_retriever = HyDERetriever(
    hypothesis_chain=hyde_hypothesis_chain,
    vector_retriever=vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4}),
)

In [ ]:
hyde_query = "How did the authors let the model look at different kinds of information at the same time?"

show_ids(vector_store.as_retriever(search_kwargs={"k": 4}).invoke(hyde_query), "Plain dense search with the raw question:")

hyde_documents = hyde_retriever.invoke(hyde_query)

print("\nOriginal query:\n", hyde_query)
print("\nHypothetical document used for retrieval:\n", hyde_retriever.last_hypothetical_document)

display_documents(hyde_documents, title="HyDE Retrieval Results", max_documents=4)

## Topic 31: HyDE with LangChain's Built-in Helper

`HypotheticalDocumentEmbedder` is **not a retriever** — it is an **embeddings wrapper**. When you embed a query with it, it first asks the LLM for a hypothetical document and then embeds that document.

So we open the **same Chroma collection** but pass the HyDE wrapper as the embedding function. Now every query is automatically HyDE-transformed before searching. No new data is embedded.

> **Note:** Never use the HyDE wrapper when *building* the index. Real chunks must be embedded with the normal `embeddings` object (Topic 10). The wrapper is used **only for queries**.

In [ ]:
built_in_hyde_embeddings = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=embeddings,
    prompt_key="web_search",  # one of LangChain's ready-made HyDE prompts
)

built_in_hyde_vector_store = Chroma(
    collection_name=COLLECTION_NAME,                 # SAME collection as Topic 10
    embedding_function=built_in_hyde_embeddings,     # only the query side changes
    persist_directory=str(PERSIST_DIRECTORY),
)

built_in_hyde_retriever = built_in_hyde_vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

built_in_hyde_documents = built_in_hyde_retriever.invoke(hyde_query)

display_documents(built_in_hyde_documents, title="Built-in LangChain HyDE Results", max_documents=4)

---
# Part E — Post-Retrieval (after retrieval, before the LLM)

The retriever gives us candidate chunks. Before sending them to the LLM we can make them **better**:
- **Compression** → cut out the irrelevant sentences, or drop weak chunks completely (32–34).
- **Reranking** → a more accurate model re-orders the candidates and keeps only the best (35–37).

All of them use `ContextualCompressionRetriever`:

```
ContextualCompressionRetriever = base_retriever (finds candidates) + base_compressor (cleans / filters / reorders them)
```

## Topic 32: Contextual Compression — LLM Extractor

**Problem:** a retrieved chunk is ~1000 characters, but maybe only one sentence answers the question. The rest is noise that wastes tokens and can confuse the LLM.

**`LLMChainExtractor`:** for **every** retrieved chunk, the LLM is asked *"copy only the parts of this context that are relevant to the question, AS IS"*. If nothing is relevant, it returns `NO_OUTPUT` and the chunk is dropped.

| ✅ Pros | ❌ Cons |
|---|---|
| very clean, short context | **one LLM call per chunk** → slow and costs tokens |
| drops useless chunks | the LLM can occasionally cut something useful |

In [ ]:
compression_base_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

compression_query = "How is the attention function computed in scaled dot-product attention?"

base_results = compression_base_retriever.invoke(compression_query)
display_documents(base_results, title="BEFORE compression (full chunks)", max_characters=1000)

In [ ]:
llm_extractor = LLMChainExtractor.from_llm(llm)

llm_compression_retriever = ContextualCompressionRetriever(
    base_compressor=llm_extractor,
    base_retriever=compression_base_retriever,
)

compressed_results = llm_compression_retriever.invoke(compression_query)

for document in base_results:
    print("Original length:", len(document.page_content))
for document in compressed_results:
    print("Compressed length:", len(document.page_content))

display_documents(compressed_results, title="AFTER LLM extraction (only relevant parts)", max_characters=1000)

## Topic 33: Embeddings Filter & Compressor Pipeline

**`EmbeddingsFilter`** — a cheap filter, **no LLM**. It embeds the query and every retrieved chunk and **drops whole chunks** whose similarity is below `similarity_threshold`. The score is kept on each returned document in `document.state["query_similarity_score"]`.

> The class notebook used `0.732` because OpenAI embeddings give high scores. MiniLM scores are lower (look at Topic 14), so we use **0.45**. Always tune this for your embedding model.

**`DocumentCompressorPipeline`** — chain several compressors. Put the **cheap** one first, the **expensive** one last:

```
retrieved chunks ──► EmbeddingsFilter (cheap, drops weak chunks) ──► LLMChainExtractor (expensive, only on survivors)
```
This means fewer LLM calls.

In [ ]:
EMBEDDING_FILTER_THRESHOLD = 0.45  # tune after looking at the scores

embeddings_filter = EmbeddingsFilter(
    embeddings=embeddings,
    similarity_threshold=EMBEDDING_FILTER_THRESHOLD,
)

filter_compression_retriever = ContextualCompressionRetriever(
    base_compressor=embeddings_filter,
    base_retriever=vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 6}),
)

filtered_results = filter_compression_retriever.invoke(compression_query)
print(f"Kept {len(filtered_results)} of 6 chunks (similarity >= {EMBEDDING_FILTER_THRESHOLD})")
for document in filtered_results:
    score = getattr(document, "state", {}).get("query_similarity_score")
    print(f"  {document.metadata.get('chunk_id')}  similarity = {score:.4f}")

display_documents(filtered_results, title="EmbeddingsFilter Results", max_characters=300)

In [ ]:
pipeline_compressor = DocumentCompressorPipeline(
    transformers=[embeddings_filter, llm_extractor]  # cheap first, expensive second
)

pipeline_compression_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor,
    base_retriever=vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 6}),
)

pipeline_results = pipeline_compression_retriever.invoke(compression_query)
display_documents(pipeline_results, title="Pipeline: EmbeddingsFilter -> LLMChainExtractor", max_characters=800)

## Topic 34: Custom Contextual Compression

The same LLM-extractor idea built by hand:

1. `CustomLLMExtractor` → a small class with the **same prompt** LangChain's `LLMChainExtractor` uses. `compress_documents()` runs the chain on every chunk and keeps the result unless it is `NO_OUTPUT`.
2. `CustomContextualCompressionRetriever` → a custom retriever: **Step 1** base retrieval, **Step 2** compression.

The original metadata (page, section, chunk id) is kept on the compressed chunks, so citations still work.

In [ ]:
class CustomLLMExtractor:
    # Extract the relevant parts of each document with an LLM chain.

    def __init__(self, llm_chain):
        self.llm_chain = llm_chain

    @classmethod
    def from_llm(cls, llm):
        prompt = ChatPromptTemplate.from_template(
            "Given the following question and context, extract any part of the context "
            "*AS IS* that is relevant to answer the question. If none of the context is "
            "relevant return NO_OUTPUT.\n\n"
            "Remember, *DO NOT* edit the extracted parts of the context.\n\n"
            "> Question: {question}\n"
            "> Context:\n>>>\n{context}\n>>>\n"
            "Extracted relevant parts:"
        )
        return cls(llm_chain=prompt | llm | StrOutputParser())

    def compress_documents(self, documents: List[Document], query: str) -> List[Document]:
        compressed = []
        for document in documents:
            result = self.llm_chain.invoke({"question": query, "context": document.page_content}).strip()
            if result and result != "NO_OUTPUT":
                compressed.append(Document(page_content=result, metadata=document.metadata))
        return compressed


class CustomContextualCompressionRetriever(BaseRetriever):
    # Base retrieval followed by LLM compression.

    base_retriever: Any
    base_compressor: Any

    def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun) -> List[Document]:
        documents = self.base_retriever.invoke(query)                       # Step 1: retrieve
        return self.base_compressor.compress_documents(documents, query)    # Step 2: compress

In [ ]:
custom_compression_retriever = CustomContextualCompressionRetriever(
    base_retriever=compression_base_retriever,
    base_compressor=CustomLLMExtractor.from_llm(llm),
)

custom_compressed_results = custom_compression_retriever.invoke(compression_query)
display_documents(custom_compressed_results, title="Custom Contextual Compression", max_characters=800)

## Topic 35: Reranking with a Cross-Encoder

**Two-stage retrieval:**
1. **Stage 1 (fast, broad):** get many candidates, e.g. top **20**, with vector search.
2. **Stage 2 (slow, precise):** a **reranker** reads each (query, chunk) pair carefully and re-orders them. Keep only the best **5**.

**Bi-encoder vs Cross-encoder:**
| | **Bi-encoder** (embeddings) | **Cross-encoder** (reranker) |
|---|---|---|
| How | query and chunk are embedded **separately**, then compared | query and chunk are read **together** in one model |
| Speed | very fast (chunk vectors are pre-computed) | slow (must run for every pair) |
| Accuracy | good | **better**, it sees how query words relate to chunk words |
| Used for | searching millions of chunks | re-ordering a small candidate list |

That's why we use the fast one first and the accurate one second.

**Model:** `cross-encoder/ms-marco-MiniLM-L6-v2` (free, runs locally on CPU). The reranker is just another **compressor** for `ContextualCompressionRetriever`, which keeps only the `top_n=5` best chunks.

> The class notebook also showed **Cohere Rerank**. It is a hosted API that needs a Cohere key, so it is replaced here by free local rerankers (this one and FlashRank in Topic 36). The code shape is identical — only the compressor changes.

```
User query → Dense retriever → Top 20 candidates → Cross-Encoder reranker → Final top 5
```

In [ ]:
candidate_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 20},
)

cross_encoder = HuggingFaceCrossEncoder(
    model_name="cross-encoder/ms-marco-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)

cross_encoder_reranker = CrossEncoderReranker(
    model=cross_encoder,
    top_n=5,
)

reranking_retriever = ContextualCompressionRetriever(
    base_retriever=candidate_retriever,
    base_compressor=cross_encoder_reranker,
)

In [ ]:
reranking_query = "On what hardware and for how long was the Transformer trained?"

initial_candidates = candidate_retriever.invoke(reranking_query)
show_ids(initial_candidates[:10], "BEFORE reranking (vector search order, top 10 of 20):")

reranked_documents = reranking_retriever.invoke(reranking_query)
display_documents(reranked_documents, title="AFTER cross-encoder reranking: Final Top 5", max_documents=5)

## Topic 36: FlashRank Reranker

**FlashRank** is another free, local reranker. It uses small, optimized (ONNX) cross-encoder models, so it is **very fast on CPU** and has no PyTorch dependency at query time.

- Model: `ms-marco-MiniLM-L-12-v2` (downloaded automatically on first use).
- `top_n=5` → keep the 5 best chunks.
- It writes its score into `metadata["relevance_score"]`.

**Cross-encoder vs FlashRank:** both are cross-encoders; FlashRank is a lighter runtime. Compare the two top-5 lists below — they usually agree on most chunks.

In [ ]:
# ===================== Topic 36: FlashRank Reranker (full fixed cell) =====================
import sys
import subprocess
import importlib

# Step 1: make sure flashrank is installed in THIS kernel's environment
try:
    import flashrank
except ImportError:
    print("flashrank not found -> installing into:", sys.executable)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "flashrank"])
    importlib.invalidate_caches()
    import flashrank

from flashrank import Ranker, RerankRequest

# Step 2: fix the "FlashrankRerank is not fully defined" error
# (langchain_community was imported before flashrank was available, so these names are missing there)
import langchain_community.document_compressors.flashrank_rerank as fr_module

fr_module.Ranker = Ranker
fr_module.RerankRequest = RerankRequest

from langchain_community.document_compressors import FlashrankRerank
FlashrankRerank.model_rebuild(force=True)

print("FlashRank ready. Python:", sys.executable)

# Step 3: FlashRank reranker (first run downloads the model once)
flashrank_reranker = FlashrankRerank(model="ms-marco-MiniLM-L-12-v2", top_n=5)

flashrank_retriever = ContextualCompressionRetriever(
    base_retriever=candidate_retriever,
    base_compressor=flashrank_reranker,
)

# Step 4: run and compare with the cross-encoder (Topic 35)
flashrank_documents = flashrank_retriever.invoke(reranking_query)
display_documents(flashrank_documents, title="FlashRank: Final Top 5", max_documents=5, max_characters=300)

show_ids(reranked_documents, "Cross-encoder top 5:")
show_ids(flashrank_documents, "FlashRank top 5:")

## Topic 37: Hybrid Retrieval + Reranking (Best Pipeline)

This combines the best of everything:
- **BM25** catches exact keywords,
- **Dense** catches meaning,
- **RRF** merges them into one candidate list,
- **Cross-encoder** picks the truly best chunks.

```
                    ┌── BM25 Search ───────┐
User query ─────────┤                      ├── Weighted RRF
                    └── Dense Search ──────┘
                                              ↓
                                     Candidate documents
                                              ↓
                                  Cross-Encoder Reranker
                                              ↓
                                     Final top documents
```

This "hybrid + rerank" setup is very common in production RAG systems. We use it as the default retriever for generation in Part F.

In [ ]:
bm25_candidate_retriever = BM25Retriever.from_documents(chunks)
bm25_candidate_retriever.k = 15

dense_candidate_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 15},
)

hybrid_candidate_retriever = EnsembleRetriever(
    retrievers=[bm25_candidate_retriever, dense_candidate_retriever],
    weights=[0.5, 0.5],
)

hybrid_reranking_retriever = ContextualCompressionRetriever(
    base_retriever=hybrid_candidate_retriever,
    base_compressor=cross_encoder_reranker,
)

query = "What results did the Transformer achieve on English constituency parsing?"
final_documents = hybrid_reranking_retriever.invoke(query)

display_documents(final_documents, title="Hybrid Retrieval + Cross-Encoder Reranking", max_documents=5)

---
# Part F — Generation (Augmentation + Answer)

Until now we only **retrieved**. RAG = **R**etrieval + **A**ugmentation + **G**eneration:

| Step | What happens |
|---|---|
| **Retrieval** | a retriever finds relevant chunks (Parts C–E) |
| **Augmentation** | the chunks are joined into one **context** string and put into the prompt |
| **Generation** | the LLM answers **only from that context** |

Rules in the prompt:
- use **only** the context → fewer hallucinations,
- if the answer is not in the context, say **"I don't know"**,
- cite pages like **[Page 5]** → the user can check the answer. Our `format_docs()` helper (Topic 5) adds the page labels.

## Topic 38: Augmentation + Generation

First step by step (so you can see the context), then the same thing as one **LCEL chain**:

```
question ──► retriever ──► format_docs ──► prompt(context, question) ──► llm ──► StrOutputParser ──► answer
```

`RunnablePassthrough()` passes the question through unchanged, while the `retriever | format_docs` branch builds the context in parallel.

In [ ]:
rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant answering questions about the paper 'Attention Is All You Need'.\n"
        "Use ONLY the context below. Be clear, concise and accurate.\n"
        "Cite the page numbers you used, like [Page 5].\n"
        "If the answer is not in the context, say \"I don't know\" - do not make up an answer.",
    ),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

generation_question = "Why is self-attention faster to train than recurrent layers?"

# Step 1 - Retrieval
retrieved_documents = hybrid_reranking_retriever.invoke(generation_question)
show_ids(retrieved_documents, "Retrieved chunks:")

# Step 2 - Augmentation
context = format_docs(retrieved_documents)
print("\nContext sent to the LLM (first 800 characters):\n")
print(context[:800])

In [ ]:
# Step 3 - Generation
generation_chain = rag_prompt | llm | StrOutputParser()

answer = generation_chain.invoke({"context": context, "question": generation_question})

print("Question:", generation_question)
print("\nAnswer:\n", answer)

In [ ]:
# The same pipeline as one LCEL chain
rag_chain = (
    {
        "context": hybrid_reranking_retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("What is the dimensionality of the model and of the feed-forward inner layer in the base Transformer?"))

In [ ]:
# A question the paper cannot answer -> the model should say it doesn't know
print(rag_chain.invoke("What does the paper say about reinforcement learning from human feedback?"))

## Topic 39: Comparing Retrievers End-to-End

The generation step does not care **how** the documents were found — every retriever has the same `.invoke()` interface. So we can plug any retriever from this notebook into the same RAG chain and compare the answers.

This replaces the separate "generation" sections of the class notebooks (RAG Fusion pipeline, RAG Fusion + Ensemble pipeline, HyDE pipeline): it is one helper function instead of three copies of the same code.

In [ ]:
def build_rag_chain(retriever):
    # Same prompt + LLM for every retriever, only the retrieval step changes.
    return (
        {"context": retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}
        | rag_prompt
        | llm
        | StrOutputParser()
    )


retrievers_to_compare = {
    "Dense (similarity)": vector_store.as_retriever(search_kwargs={"k": 4}),
    "Similarity + MMR ensemble": similarity_mmr_ensemble,
    "RAG Fusion": rag_fusion_retriever,
    "HyDE": hyde_retriever,
    "Hybrid + Rerank": hybrid_reranking_retriever,
}

comparison_question = "How does the Transformer inject information about the position of tokens?"

for name, retriever in retrievers_to_compare.items():
    print("=" * 90)
    print(name)
    print("=" * 90)
    print(build_rag_chain(retriever).invoke(comparison_question))
    print()

---
# Part G — Agentic RAG (LangGraph)

In Part F the pipeline was **fixed**: always retrieve → always generate. An **agentic** RAG system lets the LLM **decide**:
- *Do I even need to search?* (conditional retrieval)
- *Which tool: the paper or the web?* (tool use)
- *Was the search good enough, or should I search again with a better query?* (evaluation + rewriting)
- *Should I split this question first?* (decomposition)

We build it **step by step**. Each topic adds one new capability on top of the previous one, reusing the same `vector_store` and the same node functions.

| Topic | New capability | Graph |
|---|---|---|
| 41 | Base RAG as a graph | `retrieve → generate` |
| 42 | Conditional retrieval | `route_question → (retrieve) → generate` |
| 43 | Tool use (paper / web) | `route_question → agent ⇄ tools → collect → generate` |
| 44 | Multi-round limit | + `check_retrieval_limit` |
| 45 | Evaluation + query rewriting | + `evaluate_docs → rewrite_query` loop |
| 46 | Query decomposition | + `check_decomposition → decompose_query` |
| 47 | Full agentic RAG | + `build_prompt` (dynamic system prompt) |

## Topic 40: LangGraph Basics & Shared Pieces

**LangGraph in 5 words each:**
| Concept | Meaning |
|---|---|
| **State** | a dict shared by all nodes (`query`, `context`, `generation`, ...) |
| **Node** | a Python function: reads state, returns a dict of updates |
| **Edge** | fixed arrow: after node A always go to node B |
| **Conditional edge** | a function that looks at the state and returns the **name** of the next node |
| `START` / `END` | where the graph begins and stops |

`MessagesState` is a ready-made state with a `messages` list. New messages returned by a node are **appended** (not replaced) — needed for tool calling.

> **Important LangGraph detail:** a node only *receives* the fields of the state class written in its **type hint**. If `agent(state: ToolUseRAGState)` reads a field that `ToolUseRAGState` does not declare, it gets `None` — even if another node wrote that field. So every field a node reads must exist in the state class of its type hint. Because our states **inherit** from each other, a field declared early is available to every later graph.

**Shared pieces defined once here:**
- `agent_retriever` → the retriever the graphs use (dense search on our `vector_store`; you can swap in `hybrid_reranking_retriever` from Topic 37),
- `make_initial_state()` → builds the starting state so we don't repeat long dicts,
- `show_graph()` → draws the graph (falls back to Mermaid text if drawing needs internet and fails),
- `print_result()` → prints the important fields of a finished run.

In [ ]:
AGENT_K = 4
MAX_REWRITES = 2   # the class notebook used 3; 2 saves Groq requests

agent_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": AGENT_K})


def make_initial_state(query: str, **extra) -> dict:
    # Starting state for every graph run. Extra fields (limits, counters) are passed as keyword arguments.
    state = {"query": query, "messages": [], "retrieved_docs": [], "context": ""}
    state.update(extra)
    return state


def show_graph(graph) -> None:
    # Draw the graph as an image; if that is not possible (no internet), print Mermaid text instead.
    try:
        from IPython.display import Image, display
        display(Image(graph.get_graph().draw_mermaid_png()))
    except Exception:
        print(graph.get_graph().draw_mermaid())


def print_result(result: dict, title: str) -> None:
    print("=" * 90)
    print(title)
    print("=" * 90)
    for key in ["needs_retrieval", "needs_decomposition", "sub_queries", "retrieval_count", "rewrite_count", "rewritten_query", "is_relevant"]:
        if key in result and result[key] not in (None, "", []):
            print(f"{key:20}: {result[key]}")
    print(f"{'retrieved_docs':20}: {len(result.get('retrieved_docs') or [])}")
    if "relevant_docs" in result:
        print(f"{'relevant_docs':20}: {len(result.get('relevant_docs') or [])}")
    print("\nGeneration:\n", result.get("generation"))

## Topic 41: Base RAG as a Graph

The simplest graph: exactly the Part F pipeline, but as LangGraph nodes.

```
START ──► retrieve ──► generate ──► END
```

- **`retrieve`** → runs the retriever, saves `retrieved_docs` and the formatted `context`.
- **`generate`** → answers from the context. It already has an `else` branch (answer from general knowledge when there is **no** context) — we need that from Topic 42 on.

In [ ]:
class BaseRAGState(MessagesState):
    query: str
    retrieved_docs: list[Document]
    context: str
    generation: str


answer_from_context_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using only the context below. Cite pages like [Page 5]. "
               "If the context does not contain the answer, say you don't know.\n\nContext:\n{context}"),
    ("human", "{query}"),
])

general_knowledge_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the following question from your general knowledge."),
    ("human", "{query}"),
])


def retrieve(state: BaseRAGState) -> dict:
    documents = agent_retriever.invoke(state["query"])
    return {"retrieved_docs": documents, "context": format_docs(documents)}


def generate(state: BaseRAGState) -> dict:
    context = state.get("context") or ""
    if context:
        response = (answer_from_context_prompt | llm).invoke({"context": context, "query": state["query"]})
    else:
        response = (general_knowledge_prompt | llm).invoke({"query": state["query"]})
    return {"generation": response.content}


base_builder = StateGraph(BaseRAGState)
base_builder.add_node("retrieve", retrieve)
base_builder.add_node("generate", generate)
base_builder.add_edge(START, "retrieve")
base_builder.add_edge("retrieve", "generate")
base_builder.add_edge("generate", END)

base_rag_graph = base_builder.compile()
show_graph(base_rag_graph)

In [ ]:
result = base_rag_graph.invoke(make_initial_state("What is multi-head attention and why is it used?"))
print_result(result, "BASE RAG GRAPH")

print("\nRetrieved documents:")
show_ids(result["retrieved_docs"], "")

## Topic 42: Conditional Retrieval

**Problem:** the base graph searches the paper even for *"What is the capital of Pakistan?"*. That wastes time and puts useless context in the prompt.

**Solution:** a **router** node asks the LLM (structured output → `RouteDecision(needs_retrieval: bool)`) whether the question needs the document. A **conditional edge** then goes to `retrieve` or straight to `generate`.

```
START ──► route_question ──┬── needs_retrieval = True ──► retrieve ──► generate ──► END
                           └── needs_retrieval = False ─────────────► generate ──► END
```

The new state **inherits** from `BaseRAGState` and only adds the new field — you can see the state growing topic by topic.

In [ ]:
class ConditionalRAGState(BaseRAGState):
    needs_retrieval: bool


class RouteDecision(BaseModel):
    needs_retrieval: bool = Field(
        description="True if the question needs information from the 'Attention Is All You Need' paper "
                    "or from the web (current or specific facts). False if general knowledge is enough."
    )


route_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify whether the following question requires retrieving information from a specialized "
               "document (the research paper 'Attention Is All You Need') or the web, "
               "or can be answered from your own general knowledge."),
    ("human", "{query}"),
])


def route_question(state: ConditionalRAGState) -> dict:
    decision = (route_prompt | llm.with_structured_output(RouteDecision)).invoke({"query": state["query"]})
    return {"needs_retrieval": decision.needs_retrieval}


def route_to_retrieve_or_generate(state: ConditionalRAGState) -> Literal["retrieve", "generate"]:
    return "retrieve" if state["needs_retrieval"] else "generate"


conditional_builder = StateGraph(ConditionalRAGState)
conditional_builder.add_node("route_question", route_question)
conditional_builder.add_node("retrieve", retrieve)
conditional_builder.add_node("generate", generate)
conditional_builder.add_edge(START, "route_question")
conditional_builder.add_conditional_edges("route_question", route_to_retrieve_or_generate)
conditional_builder.add_edge("retrieve", "generate")
conditional_builder.add_edge("generate", END)

conditional_rag_graph = conditional_builder.compile()
show_graph(conditional_rag_graph)

In [ ]:
# Query 1: about the paper -> should retrieve
result_domain = conditional_rag_graph.invoke(make_initial_state("What does the paper say about the number of attention heads?"))
print_result(result_domain, "QUERY 1 (paper question)")

# Query 2: general knowledge -> should skip retrieval
result_general = conditional_rag_graph.invoke(make_initial_state("What is the capital of Pakistan?"))
print_result(result_general, "QUERY 2 (general knowledge)")

## Topic 43: Tool-Use Retrieval (Agent chooses the tool)

Now the LLM becomes an **agent** with **tools**. It decides *which* tool to call, *with what query*, and *how many results* (`k`).

**Tools** (created with `@tool`):
| Tool | When the agent should use it |
|---|---|
| `vector_store_search` | anything about the Attention paper (searches our `vector_store`) |
| `web_search` (Tavily) | current / real-time info not in the paper — **only added if `TAVILY_API_KEY` is set** |

`response_format="content_and_artifact"` → a tool returns two things:
- **content** (text) → goes back to the LLM inside a `ToolMessage`,
- **artifact** (the real `Document` objects) → kept on the message, so we can collect the documents later.

**The agent loop:**
```
START ──► route_question ──┬──► agent ──(tool calls?)──► tools ──┐
                           │      ▲                              │
                           │      └──────────────────────────────┘
                           │      │ (no more tool calls)
                           │      ▼
                           │  collect_tool_output ──► generate ──► END
                           └───────────────────────► generate ──► END
```
- `llm.bind_tools(tools)` → the LLM can now reply with **tool calls** instead of text.
- `ToolNode(tools)` → a ready-made node that runs the requested tools and returns `ToolMessage`s.
- `collect_tool_output` → takes the documents from the `ToolMessage` artifacts of the **latest round** and builds the context.

**Fix compared to the class notebook:** the class `agent` node created the system + user messages but did not save them in the state, so on the next loop the LLM only saw tool results without the question. Here the starting messages are saved together with the first response.

In [ ]:
@tool(response_format="content_and_artifact")
def vector_store_search(query: str, k: int = 4):
    '''Search the research paper 'Attention Is All You Need' for relevant passages.
    Adjust k (default 4) to retrieve more or fewer passages.'''
    documents = vector_store.similarity_search(query, k=k)
    return "## Paper Results\n\n" + format_docs(documents), documents


tools = [vector_store_search]

if TAVILY_AVAILABLE:
    from tavily import TavilyClient

    @tool(response_format="content_and_artifact")
    def web_search(query: str, max_results: int = 3):
        '''Search the web for current or real-time information.
        Adjust max_results (default 3) to control how many results are returned.'''
        client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
        response = client.search(query, max_results=max_results)
        documents = [
            Document(page_content=r["content"], metadata={"source": r["url"], "title": r.get("title", "")})
            for r in response["results"]
        ]
        content = "## Web Search Results\n\n" + "\n\n".join(f"[{d.metadata['source']}]\n{d.page_content}" for d in documents)
        return content, documents

    tools.append(web_search)

print("Agent tools:", [t.name for t in tools])

In [ ]:
PAPER_DESCRIPTION = (
    "the research paper 'Attention Is All You Need' (Vaswani et al., 2017), which introduces the Transformer: "
    "encoder-decoder architecture, scaled dot-product and multi-head attention, positional encoding, "
    "why self-attention is used instead of recurrence or convolution, the training setup (data, hardware, "
    "optimizer, regularization) and results on WMT 2014 translation and English constituency parsing"
)

AGENT_SYSTEM_PROMPT = (
    "You are a retrieval agent.\n\n"
    f"- vector_store_search: use it for anything that can be answered from {PAPER_DESCRIPTION}. "
    "You may increase k if broader coverage of the paper is needed.\n"
)
if TAVILY_AVAILABLE:
    AGENT_SYSTEM_PROMPT += (
        "- web_search: use it for current or real-time information that is not in the paper "
        "(recent models, news, prices, weather). Rephrase the query into short search keywords first.\n"
    )
AGENT_SYSTEM_PROMPT += (
    "\nYou may call one tool, several tools, or the same tool more than once. "
    "When the query contains numbered steps (Step 1, Step 2, ...), make at least one retrieval call for every step. "
    "When you have enough information, reply with a short note saying you are done (do not write the final answer)."
)

agent_llm_with_tools = llm.bind_tools(tools)
tool_node = ToolNode(tools)

print(AGENT_SYSTEM_PROMPT)

In [ ]:
class ToolUseRAGState(ConditionalRAGState):
    # The tool calls live in `messages`.
    # retrieval_task stays empty until Topic 46 (decomposition) fills it. It is declared HERE because
    # LangGraph only passes a node the fields of the state class in its type hint (agent -> ToolUseRAGState).
    retrieval_task: str


def agent(state: ToolUseRAGState) -> dict:
    messages = state["messages"]
    new_messages = []

    if not messages:
        # First call: start the conversation. retrieval_task is used later by decomposition (Topic 46).
        task = state.get("retrieval_task") or state["query"]
        new_messages = [SystemMessage(content=AGENT_SYSTEM_PROMPT), HumanMessage(content=task)]
        messages = new_messages

    response = agent_llm_with_tools.invoke(messages)
    return {"messages": [*new_messages, response]}   # save the start messages too (bug fix)


def collect_tool_output(state: ToolUseRAGState) -> dict:
    # Collect documents from the ToolMessages of the latest round (everything after the last HumanMessage).
    documents = []
    for message in reversed(state["messages"]):
        if isinstance(message, HumanMessage):
            break
        if isinstance(message, ToolMessage) and message.artifact:
            documents.extend(message.artifact)
    documents.reverse()
    documents = deduplicate_documents(documents)
    return {"retrieved_docs": documents, "context": format_docs(documents)}


def route_to_agent_or_generate(state: ToolUseRAGState) -> Literal["agent", "generate"]:
    return "agent" if state["needs_retrieval"] else "generate"


def route_after_agent(state: ToolUseRAGState) -> Literal["tools", "collect_tool_output"]:
    last_message = state["messages"][-1]
    return "tools" if getattr(last_message, "tool_calls", None) else "collect_tool_output"


tool_builder = StateGraph(ToolUseRAGState)
tool_builder.add_node("route_question", route_question)
tool_builder.add_node("agent", agent)
tool_builder.add_node("tools", tool_node)
tool_builder.add_node("collect_tool_output", collect_tool_output)
tool_builder.add_node("generate", generate)

tool_builder.add_edge(START, "route_question")
tool_builder.add_conditional_edges("route_question", route_to_agent_or_generate)
tool_builder.add_conditional_edges("agent", route_after_agent)
tool_builder.add_edge("tools", "agent")
tool_builder.add_edge("collect_tool_output", "generate")
tool_builder.add_edge("generate", END)

tool_use_graph = tool_builder.compile()
show_graph(tool_use_graph)

In [ ]:
def show_tool_calls(result: dict) -> None:
    for message in result["messages"]:
        for call in getattr(message, "tool_calls", None) or []:
            print(f"  tool call -> {call['name']}({call['args']})")


# Case 1: paper only
result_vs = tool_use_graph.invoke(make_initial_state("According to the paper, what BLEU score did the Transformer achieve on English-to-French?"))
print_result(result_vs, "CASE 1: Paper question")
show_tool_calls(result_vs)

# Case 2: no retrieval needed
result_none = tool_use_graph.invoke(make_initial_state("Explain the difference between supervised and unsupervised learning."))
print_result(result_none, "CASE 2: General knowledge (no tools)")

In [ ]:
# Case 3 and 4 need the web tool
if TAVILY_AVAILABLE:
    result_web = tool_use_graph.invoke(make_initial_state("What is the latest stable version of the Python programming language?"))
    print_result(result_web, "CASE 3: Web only")
    show_tool_calls(result_web)

    result_both = tool_use_graph.invoke(make_initial_state(
        "Compare the training cost reported in the Attention paper with how long it takes to train today's largest language models."
    ))
    print_result(result_both, "CASE 4: Paper + web")
    show_tool_calls(result_both)
else:
    print("Tavily key not set -> web-search cases skipped. The agent can only use the paper.")

## Topic 44: Multi-Round Retrieval with a Limit

The agent can loop `agent → tools → agent → tools → ...` as long as it wants. That is powerful (it can search again to fill gaps) but dangerous: it may loop many times and burn API calls.

**Solution:** count the tool rounds and stop at `max_retrieval_steps`.
- `check_retrieval_limit` → `retrieval_count + 1` after every tools round.
- `route_after_limit_check` → back to `agent` while under the limit, otherwise go straight to `collect_tool_output`.

```
agent ──► tools ──► check_retrieval_limit ──┬── count < max ──► agent
                                            └── count ≥ max ──► collect_tool_output ──► generate
```

In [ ]:
class LimitedToolUseState(ToolUseRAGState):
    retrieval_count: int       # incremented after each tool-call round
    max_retrieval_steps: int   # cap on tool-call rounds before forcing generate


def check_retrieval_limit(state: LimitedToolUseState) -> dict:
    return {"retrieval_count": state.get("retrieval_count", 0) + 1}


def route_after_limit_check(state: LimitedToolUseState) -> Literal["agent", "collect_tool_output"]:
    return "collect_tool_output" if state["retrieval_count"] >= state["max_retrieval_steps"] else "agent"


limited_builder = StateGraph(LimitedToolUseState)
limited_builder.add_node("route_question", route_question)
limited_builder.add_node("agent", agent)
limited_builder.add_node("tools", tool_node)
limited_builder.add_node("check_retrieval_limit", check_retrieval_limit)
limited_builder.add_node("collect_tool_output", collect_tool_output)
limited_builder.add_node("generate", generate)

limited_builder.add_edge(START, "route_question")
limited_builder.add_conditional_edges("route_question", route_to_agent_or_generate)
limited_builder.add_conditional_edges("agent", route_after_agent)
limited_builder.add_edge("tools", "check_retrieval_limit")
limited_builder.add_conditional_edges("check_retrieval_limit", route_after_limit_check)
limited_builder.add_edge("collect_tool_output", "generate")
limited_builder.add_edge("generate", END)

limited_tool_graph = limited_builder.compile()
show_graph(limited_tool_graph)

In [ ]:
# A query designed to trigger several retrieval rounds
query_multi = (
    "Step 1: Find how scaled dot-product attention is computed. "
    "Step 2: Find how multi-head attention uses it. "
    "Step 3: Find which optimizer and learning-rate schedule were used for training. "
    "Then summarise all three."
)

result_multi = limited_tool_graph.invoke(make_initial_state(query_multi, retrieval_count=0, max_retrieval_steps=3))
print_result(result_multi, "MULTI-ROUND RETRIEVAL (max 3 rounds)")
show_tool_calls(result_multi)

## Topic 45: Retrieval Evaluation & Query Rewriting (Corrective RAG)

**Problem:** the agent may retrieve chunks that look related but don't actually answer the question. Generating from them gives a weak or wrong answer.

**Solution — check before answering:**
1. **`evaluate_docs`** → the LLM grades **each** retrieved document: `RelevanceEvaluation(is_relevant: bool)`. Only relevant ones go into `relevant_docs` and the context.
2. If **none** are relevant → **`rewrite_query`** asks the LLM for a better query (more specific, different terms) and the agent searches again.
3. After `MAX_REWRITES` failed attempts → a **fallback** answer: *"I could not find relevant information"* instead of guessing.

```
collect_tool_output ──► evaluate_docs ──┬── relevant docs found ──► generate ──► END
                                        └── none relevant ──► rewrite_query ──┬── tries left ──► agent (search again)
                                                                              └── no tries left ──► generate (fallback)
```

**Fix compared to the class notebook:** there, after a rewrite the agent rebuilt its messages from scratch on **every** call, so the tool results of the new round were lost and it kept calling tools until the limit. Here `rewrite_query` simply **adds a new HumanMessage** with the rewritten query to the conversation and resets the round counter. `collect_tool_output` already reads only the ToolMessages after the last HumanMessage, so each attempt is evaluated on its own documents.

> Cost note: grading is one LLM call **per document**. Keep `k` small.

In [ ]:
class CorrectiveRAGState(LimitedToolUseState):
    relevant_docs: list[Document]
    is_relevant: bool
    rewritten_query: str
    rewrite_count: int


class RelevanceEvaluation(BaseModel):
    is_relevant: bool = Field(description="True only if the passage directly helps answer the query")


relevance_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a relevance evaluator. Given a user query and a single document passage, "
               "determine whether the passage contains useful information to answer the query. "
               "Return is_relevant=True only if the passage directly addresses the query with specific facts or analysis. "
               "Return is_relevant=False if it is off-topic, too vague, or does not help answer the query."),
    ("human", "Query: {query}\n\nDocument passage:\n{doc}"),
])

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a query rewriting assistant. A retrieval system searched for information to answer the user's "
               "query but none of the retrieved documents were relevant. Rewrite the query to be more specific, use "
               "different terminology, or focus on a narrower aspect that is more likely to match content in the "
               f"knowledge source ({PAPER_DESCRIPTION}). Return only the rewritten query string."),
    ("human", "Original query: {query}\nPrevious attempt: {previous}\n\nRewritten query:"),
])

FALLBACK_ANSWER = (
    "I was unable to find relevant information to answer your query after multiple retrieval attempts. "
    "The knowledge source does not appear to contain content that addresses this question."
)


def evaluate_docs(state: CorrectiveRAGState) -> dict:
    documents = state.get("retrieved_docs") or []
    if not documents:
        return {"is_relevant": False, "relevant_docs": [], "context": ""}

    active_query = state.get("rewritten_query") or state.get("retrieval_task") or state["query"]
    grader = relevance_prompt | llm.with_structured_output(RelevanceEvaluation)

    relevant_docs = [d for d in documents if grader.invoke({"query": active_query, "doc": d.page_content}).is_relevant]

    return {
        "is_relevant": len(relevant_docs) > 0,
        "relevant_docs": relevant_docs,
        "context": format_docs(relevant_docs),
    }


def rewrite_query(state: CorrectiveRAGState) -> dict:
    new_query = (rewrite_prompt | llm | StrOutputParser()).invoke(
        {"query": state["query"], "previous": state.get("rewritten_query") or "(none)"}
    ).strip()
    retry_message = HumanMessage(
        content=f"None of the passages found so far were relevant. Search again using this rewritten query: {new_query}"
    )
    return {
        "rewritten_query": new_query,
        "rewrite_count": state.get("rewrite_count", 0) + 1,
        "retrieval_count": 0,             # fresh round budget for the new attempt
        "messages": [retry_message],      # appended to the conversation
    }


def generate_with_fallback(state: CorrectiveRAGState) -> dict:
    if state.get("rewrite_count", 0) >= MAX_REWRITES and not state.get("is_relevant"):
        return {"generation": FALLBACK_ANSWER}
    return generate(state)


def route_after_evaluation(state: CorrectiveRAGState) -> Literal["generate", "rewrite_query"]:
    return "generate" if state["is_relevant"] else "rewrite_query"


def route_after_rewrite(state: CorrectiveRAGState) -> Literal["agent", "generate"]:
    return "generate" if state.get("rewrite_count", 0) >= MAX_REWRITES else "agent"

In [ ]:
corrective_builder = StateGraph(CorrectiveRAGState)
corrective_builder.add_node("route_question", route_question)
corrective_builder.add_node("agent", agent)
corrective_builder.add_node("tools", tool_node)
corrective_builder.add_node("check_retrieval_limit", check_retrieval_limit)
corrective_builder.add_node("collect_tool_output", collect_tool_output)
corrective_builder.add_node("evaluate_docs", evaluate_docs)
corrective_builder.add_node("rewrite_query", rewrite_query)
corrective_builder.add_node("generate", generate_with_fallback)

corrective_builder.add_edge(START, "route_question")
corrective_builder.add_conditional_edges("route_question", route_to_agent_or_generate)
corrective_builder.add_conditional_edges("agent", route_after_agent)
corrective_builder.add_edge("tools", "check_retrieval_limit")
corrective_builder.add_conditional_edges("check_retrieval_limit", route_after_limit_check)
corrective_builder.add_edge("collect_tool_output", "evaluate_docs")
corrective_builder.add_conditional_edges("evaluate_docs", route_after_evaluation)
corrective_builder.add_conditional_edges("rewrite_query", route_after_rewrite)
corrective_builder.add_edge("generate", END)

corrective_rag_graph = corrective_builder.compile()
show_graph(corrective_rag_graph)


def corrective_state(query: str, max_retrieval_steps: int = 3) -> dict:
    return make_initial_state(
        query,
        relevant_docs=[],
        retrieval_count=0,
        max_retrieval_steps=max_retrieval_steps,
        rewrite_count=0,
        rewritten_query="",
    )

In [ ]:
# Case 1: specific question -> relevant docs on the first try
result_1 = corrective_rag_graph.invoke(corrective_state("What dropout rate was used for the base Transformer model?"))
print_result(result_1, "CASE 1: Specific question")

# Case 2: vague question -> may need a rewrite
result_2 = corrective_rag_graph.invoke(corrective_state("how long did they train it for?"))
print_result(result_2, "CASE 2: Vague question")

In [ ]:
# Case 3: off-topic for this paper -> rewrites, then the fallback answer
result_3 = corrective_rag_graph.invoke(corrective_state("What does the paper say about reinforcement learning from human feedback?"))
print_result(result_3, "CASE 3: Off-topic question")

# Case 4: general knowledge -> no retrieval at all
result_4 = corrective_rag_graph.invoke(corrective_state("What is the chemical symbol for lithium?"))
print_result(result_4, "CASE 4: General knowledge")

## Topic 46: Query Decomposition inside the Agent

In Topic 29 **we** decided to decompose. Here the **agent decides**:
1. **`check_decomposition`** → `DecompositionCheck(needs_decomposition: bool)`: does the question ask for several separate pieces of information?
2. **`decompose_query`** → `SubQueries(sub_queries: list[str])`, then builds a **retrieval task** like:
   ```
   You must call a retrieval tool for each of the following 2 steps before answering.
   Step 1: ...
   Step 2: ...
   ```
3. The agent starts its conversation with that task (the system prompt tells it to cover every step).

**Fix compared to the class notebook:** the class version **overwrote `query`** with the "Step 1 / Step 2" text, so the final answer was generated for that text instead of the user's real question. Here the steps go into a separate `retrieval_task` field; `query` stays the user's question.

```
route_question ──► check_decomposition ──┬── yes ──► decompose_query ──► agent ...
                                         └── no ───────────────────────► agent ...
```

In [ ]:
class DecomposingRAGState(CorrectiveRAGState):
    needs_decomposition: bool
    sub_queries: list[str]
    # retrieval_task is inherited from ToolUseRAGState (Topic 43)


class DecompositionCheck(BaseModel):
    needs_decomposition: bool = Field(description="True if the question asks for several distinct pieces of information")


class SubQueries(BaseModel):
    sub_queries: List[str] = Field(description="Focused, self-contained sub-queries, one per retrieval step")


decomposition_check_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a query analysis assistant. Determine if the user's question asks for multiple distinct "
               "pieces of information that each require separate retrieval. If yes, needs_decomposition should be True. "
               "If the question is simple or self-contained, needs_decomposition should be False."),
    ("human", "{query}"),
])

decompose_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a query decomposition assistant. Break the user's question into focused, self-contained "
               "sub-queries, one per retrieval step. Each sub-query should target a single distinct piece of information. "
               "Generate between two and four sub-queries."),
    ("human", "{query}"),
])


def check_decomposition(state: DecomposingRAGState) -> dict:
    result = (decomposition_check_prompt | llm.with_structured_output(DecompositionCheck)).invoke({"query": state["query"]})
    return {"needs_decomposition": result.needs_decomposition}


def decompose_query(state: DecomposingRAGState) -> dict:
    sub_queries = (decompose_prompt | llm.with_structured_output(SubQueries)).invoke({"query": state["query"]}).sub_queries
    steps = "\n".join(f"Step {i + 1}: {q}" for i, q in enumerate(sub_queries))
    task = (
        f"You must call a retrieval tool for each of the following {len(sub_queries)} steps "
        f"before answering. Do not skip any step.\n\n{steps}"
    )
    return {"sub_queries": sub_queries, "retrieval_task": task}


def route_to_decomposition_or_generate(state: DecomposingRAGState) -> Literal["check_decomposition", "generate"]:
    return "check_decomposition" if state["needs_retrieval"] else "generate"


def route_after_decomposition_check(state: DecomposingRAGState) -> Literal["decompose_query", "agent"]:
    return "decompose_query" if state["needs_decomposition"] else "agent"


decomposition_builder = StateGraph(DecomposingRAGState)
decomposition_builder.add_node("route_question", route_question)
decomposition_builder.add_node("check_decomposition", check_decomposition)
decomposition_builder.add_node("decompose_query", decompose_query)
decomposition_builder.add_node("agent", agent)
decomposition_builder.add_node("tools", tool_node)
decomposition_builder.add_node("check_retrieval_limit", check_retrieval_limit)
decomposition_builder.add_node("collect_tool_output", collect_tool_output)
decomposition_builder.add_node("evaluate_docs", evaluate_docs)
decomposition_builder.add_node("rewrite_query", rewrite_query)
decomposition_builder.add_node("generate", generate_with_fallback)

decomposition_builder.add_edge(START, "route_question")
decomposition_builder.add_conditional_edges("route_question", route_to_decomposition_or_generate)
decomposition_builder.add_conditional_edges("check_decomposition", route_after_decomposition_check)
decomposition_builder.add_edge("decompose_query", "agent")
decomposition_builder.add_conditional_edges("agent", route_after_agent)
decomposition_builder.add_edge("tools", "check_retrieval_limit")
decomposition_builder.add_conditional_edges("check_retrieval_limit", route_after_limit_check)
decomposition_builder.add_edge("collect_tool_output", "evaluate_docs")
decomposition_builder.add_conditional_edges("evaluate_docs", route_after_evaluation)
decomposition_builder.add_conditional_edges("rewrite_query", route_after_rewrite)
decomposition_builder.add_edge("generate", END)

decomposition_rag_graph = decomposition_builder.compile()
show_graph(decomposition_rag_graph)


def decomposition_state(query: str, max_retrieval_steps: int = 5) -> dict:
    return corrective_state(query, max_retrieval_steps) | {"sub_queries": [], "retrieval_task": ""}

In [ ]:
# Case 1: simple question -> no decomposition
result_1 = decomposition_rag_graph.invoke(decomposition_state("What is the value of d_model in the base Transformer?"))
print_result(result_1, "CASE 1: Simple question")

# Case 2: multi-part question -> decomposition expected
complex_agent_query = (
    "Explain how positional encoding works in the Transformer, "
    "and how the Transformer's English-to-German BLEU score compares to previous models."
)
result_2 = decomposition_rag_graph.invoke(decomposition_state(complex_agent_query))
print_result(result_2, "CASE 2: Multi-part question")
show_tool_calls(result_2)

## Topic 47: Full Agentic RAG (Dynamic Prompt)

The final graph adds one node: **`build_prompt`**. Instead of one fixed system prompt, it **builds the system prompt from what happened during the run**:

| Situation in the state | Added to the system prompt |
|---|---|
| question was decomposed | "synthesize one answer covering all parts" |
| query was rewritten | "the query was reformulated during retrieval" |
| docs came from the paper / web / both | "answer using context from the paper / web / both" |
| web docs used | a **Sources** list with URLs |
| no relevant docs | "answer from general knowledge if possible" |
| all rewrites failed | `FALLBACK` → fixed "could not find" answer |

Then `generate_from_prompt` uses that prompt.

```
START ► route_question ┬► check_decomposition ┬► decompose_query ┐
                       │                      └─────────────────┴► agent ⇄ tools ► check_retrieval_limit
                       │                                              │
                       │                         collect_tool_output ◄┘
                       │                                │
                       │                         evaluate_docs ┬► rewrite_query ┬► agent
                       │                                       │                └► build_prompt
                       │                                       └────────────────► build_prompt
                       └──────────────────────────────────────────────────────────► build_prompt ► generate ► END
```

In [ ]:
class FinalAgenticRAGState(DecomposingRAGState):
    constructed_prompt: str


def build_prompt(state: FinalAgenticRAGState) -> dict:
    if state.get("rewrite_count", 0) >= MAX_REWRITES and not state.get("is_relevant", False):
        return {"constructed_prompt": "FALLBACK"}

    sections = ["You are a knowledgeable assistant."]

    if state.get("needs_decomposition"):
        sections.append("The question was complex and split into sub-queries for retrieval. "
                        "Synthesize a single coherent answer covering all parts.")

    if state.get("rewritten_query"):
        sections.append("The original query was reformulated during retrieval to improve result quality.")

    context = state.get("context") or ""
    relevant_docs = state.get("relevant_docs") or []

    if context:
        web_docs = [d for d in relevant_docs if str(d.metadata.get("source", "")).startswith("http")]
        paper_docs = [d for d in relevant_docs if not str(d.metadata.get("source", "")).startswith("http")]

        if web_docs and paper_docs:
            sections.append("Answer using the following context from the paper 'Attention Is All You Need' and from web search:")
        elif web_docs:
            sections.append("Answer using the following context from web search:")
        else:
            sections.append("Answer using the following context from the paper 'Attention Is All You Need'. Cite pages like [Page 5]:")

        sections.append(f"Context:\n{context}")

        if web_docs:
            sections.append("Sources:\n" + "\n".join(f"- {d.metadata['source']}" for d in web_docs))
    else:
        sections.append("No relevant documents were found. Answer from your general knowledge if possible.")

    return {"constructed_prompt": "\n\n".join(sections)}


def generate_from_prompt(state: FinalAgenticRAGState) -> dict:
    if state.get("constructed_prompt") == "FALLBACK":
        return {"generation": FALLBACK_ANSWER}

    prompt = ChatPromptTemplate.from_messages([("system", "{constructed_prompt}"), ("human", "{query}")])
    response = (prompt | llm).invoke({"constructed_prompt": state["constructed_prompt"], "query": state["query"]})
    return {"generation": response.content}


def route_to_decomposition_or_build(state: FinalAgenticRAGState) -> Literal["check_decomposition", "build_prompt"]:
    return "check_decomposition" if state["needs_retrieval"] else "build_prompt"


def route_after_evaluation_final(state: FinalAgenticRAGState) -> Literal["build_prompt", "rewrite_query"]:
    return "build_prompt" if state["is_relevant"] else "rewrite_query"


def route_after_rewrite_final(state: FinalAgenticRAGState) -> Literal["agent", "build_prompt"]:
    return "build_prompt" if state.get("rewrite_count", 0) >= MAX_REWRITES else "agent"

In [ ]:
final_builder = StateGraph(FinalAgenticRAGState)
final_builder.add_node("route_question", route_question)
final_builder.add_node("check_decomposition", check_decomposition)
final_builder.add_node("decompose_query", decompose_query)
final_builder.add_node("agent", agent)
final_builder.add_node("tools", tool_node)
final_builder.add_node("check_retrieval_limit", check_retrieval_limit)
final_builder.add_node("collect_tool_output", collect_tool_output)
final_builder.add_node("evaluate_docs", evaluate_docs)
final_builder.add_node("rewrite_query", rewrite_query)
final_builder.add_node("build_prompt", build_prompt)
final_builder.add_node("generate", generate_from_prompt)

final_builder.add_edge(START, "route_question")
final_builder.add_conditional_edges("route_question", route_to_decomposition_or_build)
final_builder.add_conditional_edges("check_decomposition", route_after_decomposition_check)
final_builder.add_edge("decompose_query", "agent")
final_builder.add_conditional_edges("agent", route_after_agent)
final_builder.add_edge("tools", "check_retrieval_limit")
final_builder.add_conditional_edges("check_retrieval_limit", route_after_limit_check)
final_builder.add_edge("collect_tool_output", "evaluate_docs")
final_builder.add_conditional_edges("evaluate_docs", route_after_evaluation_final)
final_builder.add_conditional_edges("rewrite_query", route_after_rewrite_final)
final_builder.add_edge("build_prompt", "generate")
final_builder.add_edge("generate", END)

agentic_rag_graph = final_builder.compile()
show_graph(agentic_rag_graph)


def ask(query: str, max_retrieval_steps: int = 5) -> dict:
    # Run the full agentic RAG graph and print the result.
    result = agentic_rag_graph.invoke(decomposition_state(query, max_retrieval_steps) | {"constructed_prompt": ""})
    print_result(result, query)
    return result

In [ ]:
final_1 = ask("What are the three ways the Transformer uses multi-head attention?")
print("\nSystem prompt that was built:\n", final_1["constructed_prompt"][:600])

In [ ]:
final_2 = ask(
    "Why did the authors choose sinusoidal positional encodings instead of learned ones, "
    "and what regularization did they use during training?"
)

In [ ]:
final_3 = ask("What does the paper say about convolutional image classification on ImageNet?")   # not in the paper -> fallback
final_4 = ask("Who wrote the novel 'Pride and Prejudice'?")                                      # general knowledge

In [ ]:
if TAVILY_AVAILABLE:
    final_5 = ask("How does the original Transformer's training compute compare with the training of modern large language models?")
else:
    print("Tavily key not set -> paper + web example skipped.")

---
# Part H — Wrap-up

## Topic 48: Summary of All Techniques

| Stage | Technique | One-line idea | Best for |
|---|---|---|---|
| Ingestion | **Metadata** | label every page/chunk (section, page, year) | filtering, self-query, citations |
| Ingestion | **Chunking** | split pages into small overlapping pieces | precise search |
| Retriever | **Similarity search** | top-k closest vectors | simple, default search |
| Retriever | **MMR** | relevant **and** diverse | avoiding repeated chunks |
| Retriever | **Score threshold** | only results above a minimum score | avoiding junk context |
| Retriever | **Metadata filtering** | search only matching chunks | sections, users, dates, permissions |
| Retriever | **Sparse (BM25)** | keyword matching | exact terms, codes, abbreviations |
| Retriever | **Dense** | meaning matching | paraphrases, synonyms |
| Retriever | **Hybrid / Ensemble (RRF)** | merge several retrievers by rank | best general recall |
| Retriever | **Parent Document** | search small chunks, return big parents | precise search + full context |
| Retriever | **Self-Query** | LLM turns the question into query + filters | questions that mention sections, pages, dates |
| Query | **Query rewriting** | turn a follow-up into a standalone query | chatbots with history |
| Query | **Query expansion / MultiQuery** | many versions of one question, union of results | vocabulary mismatch |
| Query | **RAG Fusion** | many versions of one question, **RRF-ranked** | better ranking than plain union |
| Query | **Query decomposition** | split into smaller questions | complex multi-part questions |
| Query | **HyDE** | search with a fake answer passage | short questions vs long documents |
| Post-retrieval | **LLM extraction** | keep only relevant sentences | shorter, cleaner context |
| Post-retrieval | **Embeddings filter / pipeline** | drop weak chunks cheaply, then extract | fewer LLM calls |
| Post-retrieval | **Reranking (cross-encoder / FlashRank)** | accurate model re-orders candidates | precision of the final top results |
| Post-retrieval | **Hybrid + Reranking** | BM25 + dense + RRF + reranker | production RAG |
| Generation | **Grounded prompt** | answer only from context, cite pages, say "I don't know" | fewer hallucinations |
| Agentic | **Conditional retrieval** | LLM decides if search is needed | mixed general + document questions |
| Agentic | **Tool use** | LLM picks paper / web tool and query | several knowledge sources |
| Agentic | **Retrieval limit** | cap tool-call rounds | cost and loop control |
| Agentic | **Evaluation + rewriting** | grade docs, rewrite and retry, fallback | reliable answers, no guessing |
| Agentic | **Decomposition in the agent** | agent splits complex questions | multi-part questions |
| Agentic | **Dynamic prompt** | system prompt built from what happened | context-aware final answers |

## Topic 49: Homework Topics

Study these next. A short idea for each:

| Topic | Short idea |
|---|---|
| **Weighted score fusion** | Combine *scores* (not ranks) from several retrievers using weights (e.g. 0.7 dense + 0.3 BM25) after normalizing them to 0–1. Compare with RRF (Topic 20). |
| **Multi-hop retriever** | Answer needs facts from several steps: search → read → use what you found to make the next search. The agent in Topic 44 can already do a simple version. |
| **Sentence window retriever** | Search single sentences, then return the sentence plus a few sentences before and after it (a lighter cousin of the Parent Document Retriever). |
| **Contextual chunk headers** | Prepend the section title to every chunk before embedding, so chunks keep their context. |
| **RAG evaluation** | Measure retrieval (hit rate, MRR) and answers (faithfulness, answer relevance) with a small test set of questions about the paper. |
| **Memory in the agent** | Add a LangGraph checkpointer so the agent remembers earlier turns, and combine it with query rewriting (Topic 25). |
| **Streaming** | Use `graph.stream(...)` to watch every node's update live instead of waiting for the final result. |